# TheLook E-Commerce — Unified SQL, GIS, Graph & AI

| Section | Engine / Focus | Scope | Capabilities Demonstrated |
| :--- | :--- | :--- | :--- |
| **Analytics & AI** | **`SQL`** | **Q1 – Q5** | Full-funnel brand profitability, **5-yr daily rolling 14d & 30d clickstream cubes (2.42M+ events)**, **600M+ SKU-day inventory ledger**, and **6-yr rolling 90d brand stress cube** |
| **Analytics & AI** | **`GIS`** | **Q6 – Q8** | Spherical `ST_DISTANCE` vs. **1,200+ local micro-hubs**, **network-wide warehouse + micro-hub simulation**, and **multi-year rolling 300 km convex-hull catchments** |
| **Analytics & AI** | **`AI.*`** | **Q9 – Q13** | Zero-shot `TimesFM 2.5` (`AI.FORECAST`, `AI.DETECT_ANOMALIES`), 6-D `AI.KEY_DRIVERS`, `AI.IF` + `AI.CLASSIFY`, and geospatial-enriched `AI.GENERATE` |
| **Analytics & AI** | **`Graph`** | **Q14** | **8-hop standalone ISO GQL traversal** (`GRAPH ... MATCH p = ... RETURN`) across all 5 graph entity types (`DistributionCenter`, `Product`, `OrderItem`, `Order`, `User`) |
| **Analytics & AI** | **`All-in-One`** | **Q15** | **SQL + Property Graph + GIS + `AI.*` in a single query** touching all 7 dataset tables (`GRAPH_TABLE` + clickstream + warehouse stock + spherical micro-hubs & convex hulls + `AI.GENERATE`) |
| **Conversational Graph Agent** | **`Chat → Create Agent`** | **Cell 5** | **Natural-language Conversational Analytics** (`Chat` → `Create Agent`) on `ecommerce_graph` in BigQuery Studio, grounded by `OPTIONS(description=...)` and grain-safe `MEASURE(...)` definitions |
| **Storage & Telemetry** | **Audit** | **Cells 6 – 7** | **Live physical storage & table-origin check** (`0 bytes` in BigQuery) and **Cache Analysis** (`Logical Scan → Compressed → Regional Block Cache → Wire Transfer`) |
| **Knowledge Catalog** | **Catalog** | **Cell 8** | **Automatic storage-agnostic cataloging & agentic context** (zero-touch `@biglake` S3 Iceberg + `@bigquery` Property Graph discovery & live `:lookupContext` YAML) |

In [ ]:
# @title 0. Setup & Client Initialization
# @markdown Leave `PROJECT_ID` empty to auto-detect from `.env`, environment variables, or active Google Cloud / Colab Studio credentials.
PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "us-east4"  # @param {type:"string"}
CATALOG_ID = "thelook_catalog"  # @param {type:"string"}
NAMESPACE_ID = "thelook_ecommerce"  # @param {type:"string"}
GRAPH_DATASET_ID = "thelook_commerce"  # @param {type:"string"}

from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta, timezone
import html
import logging
import os
from pathlib import Path
import re
import time
import urllib.parse
import google.auth
import pandas as pd
from requests.adapters import HTTPAdapter
from google.cloud import bigquery
from IPython.display import HTML, display, update_display


def _load_dotenv_defaults() -> dict[str, str]:
    vals: dict[str, str] = {}
    for candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
        if candidate.is_file():
            for raw in candidate.read_text(encoding="utf-8").splitlines():
                line = raw.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                vals[k.strip()] = v.strip().strip('"').strip("'")
            break
    return vals


_ENV_FILE = _load_dotenv_defaults()
if not PROJECT_ID.strip():
    PROJECT_ID = (
        _ENV_FILE.get("GCP_PROJECT")
        or os.environ.get("GCP_PROJECT")
        or os.environ.get("GOOGLE_CLOUD_PROJECT")
        or os.environ.get("GCLOUD_PROJECT")
        or ""
    ).strip()
if not PROJECT_ID:
    try:
        _, _detected_proj = google.auth.default()
        PROJECT_ID = (_detected_proj or "").strip()
    except Exception:
        PROJECT_ID = ""
if not PROJECT_ID:
    try:
        PROJECT_ID = (bigquery.Client().project or "").strip()
    except Exception:
        PROJECT_ID = ""
if not PROJECT_ID:
    raise RuntimeError(
        "Could not auto-detect PROJECT_ID. Enter your Google Cloud Project ID in Cell 0 above or set GCP_PROJECT in .env."
    )

if LOCATION.strip() == "us-east4" and _ENV_FILE.get("GCP_REGION"):
    LOCATION = _ENV_FILE["GCP_REGION"].strip()
if CATALOG_ID.strip() == "thelook_catalog" and _ENV_FILE.get("FEDERATED_CATALOG"):
    CATALOG_ID = _ENV_FILE["FEDERATED_CATALOG"].strip()
if NAMESPACE_ID.strip() == "thelook_ecommerce" and _ENV_FILE.get("CATALOG_NAMESPACE"):
    NAMESPACE_ID = _ENV_FILE["CATALOG_NAMESPACE"].strip()
if GRAPH_DATASET_ID.strip() == "thelook_commerce" and _ENV_FILE.get("BQ_DEMO_DATASET"):
    GRAPH_DATASET_ID = _ENV_FILE["BQ_DEMO_DATASET"].strip()

TABLES_FQN = f"{PROJECT_ID}.{CATALOG_ID}.{NAMESPACE_ID}"
GRAPH_DATASET_FQN = f"{PROJECT_ID}.{GRAPH_DATASET_ID}"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
if not getattr(client._http, "_is_mtls", False):
    _http_adapter = HTTPAdapter(pool_connections=32, pool_maxsize=32)
    client._http.mount("https://", _http_adapter)
    client._http.mount("http://", _http_adapter)


def _job_to_df(job: bigquery.QueryJob) -> pd.DataFrame:
    res = job.result()
    try:
        return res.to_dataframe()
    except Exception:
        return pd.DataFrame([dict(row.items()) for row in res])


def _kc_api_session():
    import requests as _http_requests
    from google.auth.transport.requests import Request as _GoogleAuthRequest

    creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    sess = _http_requests.Session()
    token = ""
    if creds is not None:
        if not getattr(creds, "valid", False):
            creds.refresh(_GoogleAuthRequest())
        token = getattr(creds, "token", "") or ""
    sess.headers.update({
        "Authorization": f"Bearer {token}",
        "x-goog-user-project": PROJECT_ID,
    })
    return sess


def _strip_html_tags(raw_html: str) -> str:
    clean = re.sub(r"<[^>]+>", " ", str(raw_html or ""))
    return re.sub(r"\s+", " ", clean).strip()


def _find_aspect_data(aspects: dict, suffix: str) -> dict:
    for k, val in (aspects or {}).items():
        if k == suffix or k.endswith("." + suffix):
            return (val or {}).get("data") or {}
    return {}


def _resolve_snapshot_anchor() -> date:
    try:
        sql = f"SELECT DATE_SUB(CAST(MAX(created_at) AS DATE), INTERVAL 4 DAY) AS d FROM `{TABLES_FQN}.orders`"
        df_a = _job_to_df(client.query(sql, location=LOCATION))
        val = df_a["d"].iloc[0]
        if pd.notna(val):
            return val if isinstance(val, date) and not isinstance(val, datetime) else pd.Timestamp(val).date()
    except Exception:
        pass
    return date(2026, 6, 30)


def _resolve_table_logical_bytes() -> dict[str, int]:
    defaults = {
        "events": 384678804,
        "inventory_items": 80897145,
        "users": 19820850,
        "order_items": 13569337,
        "orders": 6728888,
        "products": 4285975,
        "distribution_centers": 809,
    }
    try:
        sql = "SELECT table_id, size_bytes FROM `bigquery-public-data.thelook_ecommerce.__TABLES__`"
        df_b = _job_to_df(client.query(sql, location="US"))
        for _, r in df_b.iterrows():
            tid = str(r["table_id"])
            sz = int(r["size_bytes"] or 0)
            if tid in defaults and sz > 0:
                defaults[tid] = sz
    except Exception:
        pass
    return defaults


ANCHOR_DATE = _resolve_snapshot_anchor()
SNAP_END = ANCHOR_DATE.isoformat()
SNAP_5Y_START = (ANCHOR_DATE - timedelta(days=1825)).isoformat()
SNAP_6_5Y_START = (ANCHOR_DATE - timedelta(days=2372)).isoformat()
SNAP_6Y_START = (ANCHOR_DATE - timedelta(days=2190)).isoformat()
SNAP_3_5Y_START = (ANCHOR_DATE - timedelta(days=1276)).isoformat()

_FORECAST_SUNDAY = ANCHOR_DATE - timedelta(days=(ANCHOR_DATE.weekday() + 1) % 7)
FORECAST_END = _FORECAST_SUNDAY.isoformat()
FORECAST_START = (_FORECAST_SUNDAY - timedelta(days=130 * 7 - 1)).isoformat()

ANOM_HIST_START = (ANCHOR_DATE - timedelta(days=545)).isoformat()
ANOM_CUTOFF = (ANCHOR_DATE - timedelta(days=60)).isoformat()

_YOY_CURR_START_OBJ = ANCHOR_DATE - timedelta(days=179)
YOY_CURR_START = _YOY_CURR_START_OBJ.isoformat()
YOY_CURR_END = SNAP_END
YOY_PREV_START = (_YOY_CURR_START_OBJ - timedelta(days=365)).isoformat()
YOY_PREV_END = (ANCHOR_DATE - timedelta(days=365)).isoformat()

TABLE_LOGICAL_BYTES = _resolve_table_logical_bytes()


ENGINE_COLORS = {
    "SQL": ("#e8f0fe", "#1967d2"),
    "GIS": ("#e6f4ea", "#137333"),
    "AI.*": ("#f3e8fd", "#7627bb"),
    "Graph": ("#fef7e0", "#b06000"),
    "All-in-One": ("#fce8e6", "#c5221f"),
}


def _render_scoreboard_html(states: list[dict], wall_elapsed: float, done: bool, batch_id: str) -> str:
    completed = sum(1 for s in states if s["status"] == "DONE")
    failed = sum(1 for s in states if s["status"] == "ERROR")
    total = len(states)
    pct = int((completed + failed) * 100 / max(total, 1))
    total_slot_sec = sum(s["slot_sec"] for s in states)
    elastic_slots = (total_slot_sec / wall_elapsed) if wall_elapsed > 0 else 0.0

    rows_html = []
    for s in states:
        bg_pill, fg_pill = ENGINE_COLORS.get(s["engine"], ("#f1f5f9", "#334155"))
        if s["status"] == "RUNNING":
            status_badge = (
                '<span style="display:inline-block;padding:3px 10px;border-radius:999px;'
                'background:#fef7e0;color:#b06000;font-weight:600;font-size:12px;">⏳ RUNNING</span>'
            )
            slot_info = "—"
            row_info = "—"
        elif s["status"] == "DONE":
            status_badge = (
                '<span style="display:inline-block;padding:3px 10px;border-radius:999px;'
                'background:#e6f4ea;color:#137333;font-weight:600;font-size:12px;">✅ DONE</span>'
            )
            avg_s = max(s["slot_sec"] / max(s["elapsed"], 0.01), 0.1)
            slot_style = "color:#0d652d;font-weight:700;" if avg_s >= 8.0 else "color:#334155;font-weight:600;"
            slot_info = (
                f'<span style="{slot_style}">{s["slot_sec"]:,.1f}s</span> '
                f'<span style="font-size:11px;color:#475569;font-weight:600;">(~{max(round(avg_s), 1):,} slots)</span>'
            )
            row_info = f'{s["row_count"]:,} rows'
        else:
            err_short = html.escape((s.get("error") or "Error")[:80])
            status_badge = (
                f'<span title="{err_short}" style="display:inline-block;padding:3px 10px;border-radius:999px;'
                'background:#fce8e6;color:#c5221f;font-weight:600;font-size:12px;">❌ ERROR</span>'
            )
            slot_info = "—"
            row_info = err_short

        rows_html.append(
            f'<tr style="border-bottom:1px solid #e2e8f0;">'
            f'<td style="padding:8px 10px;font-weight:700;color:#475569;">Q{s["id"]}</td>'
            f'<td style="padding:8px 10px;"><span style="padding:2px 8px;border-radius:6px;'
            f'background:{bg_pill};color:{fg_pill};font-weight:600;font-size:11px;">{html.escape(s["engine"])}</span></td>'
            f'<td style="padding:8px 10px;"><div style="font-weight:600;color:#0f172a;">{html.escape(s["title"])}</div>'
            f'<div style="font-size:12px;color:#64748b;">{html.escape(s["feature"])}</div></td>'
            f'<td style="padding:8px 10px;text-align:center;">{status_badge}</td>'
            f'<td style="padding:8px 10px;text-align:right;font-variant-numeric:tabular-nums;font-weight:600;color:#0f172a;">'
            f'{s["elapsed"]:.2f}s</td>'
            f'<td style="padding:8px 10px;text-align:right;font-variant-numeric:tabular-nums;">{slot_info}</td>'
            f'<td style="padding:8px 10px;text-align:right;font-variant-numeric:tabular-nums;color:#475569;">{row_info}</td>'
            f"</tr>"
        )

    header_status = (
        f"✅ All {completed}/{total} Queries Finished in <b>{wall_elapsed:.2f}s</b> Wall-Clock &nbsp;|&nbsp; "
        f"⚡ Elastic Compute Delivered: <b>{total_slot_sec:,.1f} slot-sec</b> (<b>{total_slot_sec / 60.0:.1f} CPU-min</b> → <b>~{elastic_slots:,.0f} Concurrent Slots</b>)"
        if done and failed == 0
        else f"⚡ Executing {total} BigQuery Jobs in Parallel ({completed}/{total} complete) — Wall-Clock: <b>{wall_elapsed:.2f}s</b> &nbsp;|&nbsp; Slot-Time So Far: <b>{total_slot_sec:,.1f}s</b>"
    )
    banner_bg = "#e6f4ea" if (done and failed == 0) else "#e8f0fe"
    banner_fg = "#0d652d" if (done and failed == 0) else "#174ea6"

    return f'''
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                background:#ffffff;color:#0f172a;border:1px solid #cbd5e1;border-radius:10px;
                padding:16px;margin:8px 0;box-shadow:0 1px 3px rgba(0,0,0,0.06);">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;
                  background:{banner_bg};color:{banner_fg};padding:10px 14px;border-radius:8px;margin-bottom:10px;">
        <div style="font-size:13.5px;">{header_status}</div>
        <div style="font-size:12px;opacity:0.85;">Batch: <code>{html.escape(batch_id)}</code></div>
      </div>
      <div style="height:6px;background:#e2e8f0;border-radius:999px;overflow:hidden;margin-bottom:12px;">
        <div style="width:{pct}%;height:100%;background:#1a73e8;transition:width 0.2s ease;"></div>
      </div>
      <table style="width:100%;border-collapse:collapse;font-size:13px;background:#ffffff;color:#0f172a;">
        <thead>
          <tr style="background:#f8fafc;border-bottom:2px solid #cbd5e1;text-align:left;color:#475569;font-size:12px;">
            <th style="padding:8px 10px;">#</th>
            <th style="padding:8px 10px;">Engine</th>
            <th style="padding:8px 10px;">Query & Capability</th>
            <th style="padding:8px 10px;text-align:center;">Status</th>
            <th style="padding:8px 10px;text-align:right;">Wall Time</th>
            <th style="padding:8px 10px;text-align:right;">Slot-Time (Concurrency)</th>
            <th style="padding:8px 10px;text-align:right;">Output</th>
          </tr>
        </thead>
        <tbody>
          {"".join(rows_html)}
        </tbody>
      </table>
    </div>
    '''


def run_parallel_suite(queries: list[dict]) -> tuple[list[dict], str]:
    batch_id = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
    t_wall_start = time.perf_counter()

    states = [
        {
            "id": q["id"],
            "engine": q["engine"],
            "title": q["title"],
            "feature": q["feature"],
            "sql": q["sql"],
            "status": "RUNNING",
            "t0": t_wall_start,
            "elapsed": 0.0,
            "slot_sec": 0.0,
            "row_count": 0,
            "df": None,
            "job_id": None,
            "error": None,
        }
        for q in queries
    ]

    disp_id = f"scoreboard_{batch_id}"
    display(HTML(_render_scoreboard_html(states, 0.0, False, batch_id)), display_id=disp_id)

    def _worker(idx: int) -> None:
        st = states[idx]
        st["t0"] = time.perf_counter()
        try:
            cfg = bigquery.QueryJobConfig(
                use_query_cache=False,
                labels={
                    "demo_suite": "thelook_suite",
                    "demo_batch": batch_id,
                    "query_num": f"q{st['id']}",
                },
            )
            job = client.query(st["sql"], job_config=cfg, location=LOCATION)
            st["job_id"] = job.job_id
            df = _job_to_df(job)
            job.reload()
            st["elapsed"] = time.perf_counter() - st["t0"]
            st["slot_sec"] = (job.slot_millis or 0) / 1000.0
            st["df"] = df
            st["row_count"] = len(df)
            st["status"] = "DONE"
        except Exception as exc:  # noqa: BLE001
            st["elapsed"] = time.perf_counter() - st["t0"]
            st["error"] = str(exc)
            st["status"] = "ERROR"

    with ThreadPoolExecutor(max_workers=len(queries)) as pool:
        futures = [pool.submit(_worker, i) for i in range(len(queries))]
        while not all(f.done() for f in futures):
            now = time.perf_counter()
            for st in states:
                if st["status"] == "RUNNING":
                    st["elapsed"] = now - st["t0"]
            update_display(
                HTML(_render_scoreboard_html(states, now - t_wall_start, False, batch_id)),
                display_id=disp_id,
            )
            time.sleep(0.25)

    total_wall = time.perf_counter() - t_wall_start
    update_display(
        HTML(_render_scoreboard_html(states, total_wall, True, batch_id)),
        display_id=disp_id,
    )
    return states, batch_id


def render_results_accordion(states: list[dict], default_open_ids: set[int] | None = None) -> None:
    open_ids = default_open_ids or set()
    cards = []
    for s in states:
        is_open = "open" if s["id"] in open_ids else ""
        bg_pill, fg_pill = ENGINE_COLORS.get(s["engine"], ("#f1f5f9", "#334155"))
        if s["df"] is not None:
            tbl_html = s["df"].to_html(index=False, border=0, classes="bq-demo-tbl")
        else:
            tbl_html = f'<pre style="color:#c5221f;">{html.escape(s.get("error") or "No data")}</pre>'

        avg_s = max(round(s["slot_sec"] / max(s["elapsed"], 0.01)), 1)
        cards.append(f'''
        <details {is_open} style="background:#ffffff;color:#0f172a;border:1px solid #cbd5e1;
                                  border-radius:8px;margin:10px 0;padding:10px 14px;">
          <summary style="cursor:pointer;font-weight:600;font-size:14px;color:#0f172a;outline:none;">
            <span style="padding:2px 8px;border-radius:6px;background:{bg_pill};color:{fg_pill};
                         font-size:11px;margin-right:8px;">{html.escape(s["engine"])}</span>
            Q{s["id"]}: {html.escape(s["title"])}
            <span style="float:right;font-weight:500;font-size:12px;color:#475569;">
              ⏱ {s["elapsed"]:.2f}s &nbsp;|&nbsp; ⚡ {s["slot_sec"]:,.1f}s (~{avg_s:,} slots) &nbsp;|&nbsp; {s["row_count"]:,} rows
            </span>
          </summary>
          <div style="margin-top:10px;">
            <details style="margin-bottom:10px;">
              <summary style="cursor:pointer;font-size:12px;color:#1a73e8;font-weight:600;">
                Show Executed SQL
              </summary>
              <pre style="background:#f8fafc;color:#0f172a;border:1px solid #e2e8f0;border-radius:6px;
                          padding:10px;font-size:12px;overflow-x:auto;margin-top:6px;">{html.escape(s["sql"].strip())}</pre>
            </details>
            <div style="overflow-x:auto;">{tbl_html}</div>
          </div>
        </details>
        ''')

    style = '''
    <style>
      table.bq-demo-tbl { width:100%; border-collapse:collapse; font-size:12.5px; background:#ffffff; color:#0f172a; }
      table.bq-demo-tbl th { background:#f1f5f9; color:#334155; padding:7px 10px; text-align:left; border-bottom:2px solid #cbd5e1; }
      table.bq-demo-tbl td { padding:6px 10px; border-bottom:1px solid #e2e8f0; color:#0f172a; }
      table.bq-demo-tbl tr:hover td { background:#f8fafc; }
    </style>
    '''
    display(HTML(style + "".join(cards)))


# --- Visual Insights Grid Helpers (Step 3: 15 Bespoke SVG + HTML Cards) ---
import math
import re

DC_COORDS = {
    "Memphis TN": (-89.9711, 35.1174),
    "Chicago IL": (-87.6847, 41.8369),
    "Houston TX": (-95.3698, 29.7604),
    "Los Angeles CA": (-118.2500, 34.0500),
    "New Orleans LA": (-90.0667, 29.9500),
    "Port Authority of New York/New Jersey NY/NJ": (-73.7834, 40.6340),
    "Port Authority of NY/NJ": (-73.7834, 40.6340),
    "Philadelphia PA": (-75.1667, 39.9500),
    "Mobile AL": (-88.0431, 30.6944),
    "Charleston SC": (-79.9333, 32.7833),
    "Savannah GA": (-81.1167, 32.0167),
}

MICROHUB_COORDS = {
    "Los Angeles, CA": (-118.2437, 34.0522),
    "Torrance, CA": (-118.3406, 33.8358),
    "Chula Vista, CA": (-117.0842, 32.6401),
    "San Jose, CA": (-121.8863, 37.3382),
    "San Diego, CA": (-117.1611, 32.7157),
    "San Francisco, CA": (-122.4194, 37.7749),
    "Sacramento, CA": (-121.4944, 38.5816),
    "Fresno, CA": (-119.7871, 36.7378),
    "Long Beach, CA": (-118.1937, 33.7701),
    "Anaheim, CA": (-117.9143, 33.8366),
    "Riverside, CA": (-117.3962, 33.9533),
    "Bakersfield, CA": (-119.0187, 35.3733),
    "Houston, TX": (-95.3698, 29.7604),
    "Dallas, TX": (-96.7970, 32.7767),
    "Austin, TX": (-97.7431, 30.2672),
    "San Antonio, TX": (-98.4936, 29.4241),
    "Chicago, IL": (-87.6298, 41.8781),
    "New York, NY": (-74.0060, 40.7128),
    "Miami, FL": (-80.1918, 25.7617),
    "Phoenix, AZ": (-112.0740, 33.4484),
    "Seattle, WA": (-122.3321, 47.6062),
}

STATE_CENTROIDS = {
    "California": (-119.4179, 36.7783),
    "Texas": (-99.9018, 31.9686),
    "Florida": (-81.5158, 27.6648),
    "New York": (-75.4999, 43.0000),
    "Washington": (-120.7401, 47.7511),
    "Illinois": (-89.3985, 40.6331),
    "Pennsylvania": (-77.1945, 41.2033),
    "Ohio": (-82.9071, 40.4173),
    "Georgia": (-83.4412, 32.6782),
    "Arizona": (-111.0937, 34.0489),
    "North Carolina": (-79.0193, 35.7596),
    "Michigan": (-84.5361, 43.3266),
    "New Jersey": (-74.4057, 40.0583),
    "Virginia": (-78.6569, 37.4316),
}

# Simplified Continental US perimeter in (lon, lat) for crisp inline SVG maps
US_OUTLINE_LONLAT = [
    (-124.7, 48.4), (-124.2, 46.2), (-124.5, 42.0), (-123.8, 39.0), (-122.5, 37.7),
    (-120.6, 34.5), (-118.4, 34.0), (-117.1, 32.5), (-114.6, 32.7), (-111.1, 31.3),
    (-108.2, 31.3), (-106.5, 31.8), (-104.0, 29.5), (-102.9, 29.8), (-101.4, 29.8),
    (-99.1, 26.4), (-97.2, 26.0), (-97.0, 27.8), (-95.2, 29.1), (-93.8, 29.8),
    (-91.4, 29.6), (-89.2, 29.1), (-88.8, 30.3), (-87.5, 30.3), (-85.5, 29.9),
    (-84.1, 30.1), (-82.8, 28.0), (-81.1, 25.2), (-80.1, 25.8), (-80.5, 28.5),
    (-81.4, 30.7), (-80.8, 32.1), (-78.5, 33.9), (-75.5, 35.2), (-76.0, 37.0),
    (-75.1, 38.5), (-74.0, 40.5), (-72.0, 41.0), (-70.0, 41.7), (-70.7, 43.1),
    (-68.9, 44.3), (-67.0, 44.8), (-67.8, 47.1), (-69.2, 47.4), (-71.5, 45.0),
    (-75.0, 45.0), (-76.3, 43.6), (-79.0, 43.2), (-82.5, 42.0), (-82.4, 43.0),
    (-84.0, 45.9), (-87.5, 45.1), (-87.8, 42.5), (-87.1, 41.7), (-86.5, 44.0),
    (-85.0, 45.8), (-88.0, 46.7), (-90.9, 46.9), (-92.1, 46.8), (-95.2, 49.0),
    (-104.0, 49.0), (-116.0, 49.0), (-122.8, 49.0), (-124.7, 48.4),
]


def _vf(val, default: float = 0.0) -> float:
    try:
        if val is None or pd.isna(val):
            return default
        return float(val)
    except Exception:
        return default


def _vi(val, default: int = 0) -> int:
    return int(round(_vf(val, default)))


def _vs(val, default: str = "") -> str:
    try:
        if val is None or pd.isna(val):
            return default
        return str(val)
    except Exception:
        return default


def _short_dc(name: str) -> str:
    s = _vs(name)
    if "New York" in s or "NY/NJ" in s:
        return "NY/NJ Port"
    if "Philadelphia" in s:
        return "Philadelphia PA"
    if "Los Angeles" in s:
        return "Los Angeles CA"
    return s


def _parse_wkt_point(wkt: str):
    m = re.search(r"POINT\s*\(\s*([-\d\.]+)\s+([-\d\.]+)\s*\)", _vs(wkt), flags=re.IGNORECASE)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None


def _us_project(lon: float, lat: float, width: float, height: float, pad_x: float = 20.0, pad_y: float = 16.0):
    lon_min, lon_max = -125.5, -66.5
    lat_min, lat_max = 24.5, 49.8
    x = pad_x + ((lon - lon_min) / (lon_max - lon_min)) * (width - 2 * pad_x)
    y = pad_y + ((lat_max - lat) / (lat_max - lat_min)) * (height - 2 * pad_y)
    return round(x, 1), round(y, 1)


def _us_base_svg_elements(width: float, height: float) -> str:
    pts = " ".join(f"{x},{y}" for x, y in (_us_project(lon, lat, width, height) for lon, lat in US_OUTLINE_LONLAT))
    grid = []
    for lat in (30, 37, 44):
        x1, y1 = _us_project(-124.5, lat, width, height)
        x2, y2 = _us_project(-67.5, lat, width, height)
        grid.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#e2e8f0" stroke-dasharray="3,3" stroke-width="0.8"/>')
    for lon in (-115, -100, -85, -75):
        x1, y1 = _us_project(lon, 25.5, width, height)
        x2, y2 = _us_project(lon, 48.8, width, height)
        grid.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="#e2e8f0" stroke-dasharray="3,3" stroke-width="0.8"/>')
    return (
        f'<rect x="0" y="0" width="{width}" height="{height}" rx="8" fill="#f8fafc" stroke="#e2e8f0"/>'
        + "".join(grid)
        + f'<polygon points="{pts}" fill="#eef2f6" stroke="#cbd5e1" stroke-width="1.3" stroke-linejoin="round"/>'
    )


def _viz_q1(df: pd.DataFrame):
    max_rev = max((_vf(r.get("net_revenue")) for _, r in df.iterrows()), default=1.0) or 1.0
    men = df[df["department"] == "Men"]
    women = df[df["department"] == "Women"]
    top_m = men.iloc[0] if not men.empty else df.iloc[0]
    top_w = women.iloc[0] if not women.empty else df.iloc[-1]
    best_margin_row = df.loc[df["gross_margin_pct"].astype(float).idxmax()]
    takeaway = (
        f"<b>{html.escape(_vs(top_m['brand']))}</b> leads Men (${_vf(top_m['gross_profit'])/1e3:.1f}k profit) and "
        f"<b>{html.escape(_vs(top_w['brand']))}</b> leads Women (${_vf(top_w['gross_profit'])/1e3:.1f}k), while "
        f"<b>{html.escape(_vs(best_margin_row['brand']))}</b> captures peak margin ({_vf(best_margin_row['gross_margin_pct']):.1f}%)."
    )

    sections = []
    for dept, accent, soft in (("Men", "#1967d2", "#dbeafe"), ("Women", "#7c3aed", "#ede9fe")):
        sub = df[df["department"] == dept]
        if sub.empty:
            continue
        rows = []
        for _, r in sub.iterrows():
            rev = _vf(r.get("net_revenue"))
            prof = _vf(r.get("gross_profit"))
            margin = _vf(r.get("gross_margin_pct"))
            ret = _vf(r.get("return_rate_pct"))
            views = max(_vi(r.get("clickstream_views")), 1)
            orders = _vi(r.get("items_ordered"))
            conv = orders * 100.0 / views
            rev_pct = min(100.0, rev * 100.0 / max_rev)
            prof_pct = min(100.0, prof * 100.0 / max_rev)
            ret_bg, ret_fg = ("#dcfce7", "#15803d") if ret < 8.5 else (("#fef3c7", "#b45309") if ret <= 10.0 else ("#fee2e2", "#b91c1c"))
            rows.append(
                f'<div style="display:grid;grid-template-columns:132px 1fr 138px;align-items:center;gap:8px;margin:5px 0;">'
                f'<div style="font-size:12px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">'
                f'<span style="font-weight:700;color:{accent};">#{_vi(r.get("profit_rank"))}</span> '
                f'<span style="font-weight:600;color:#0f172a;">{html.escape(_vs(r.get("brand")))}</span></div>'
                f'<div title="Net Rev: ${rev:,.0f} | Gross Profit: ${prof:,.0f} ({margin:.1f}%)" '
                f'style="position:relative;height:16px;background:#f1f5f9;border-radius:4px;overflow:hidden;">'
                f'<div style="position:absolute;left:0;top:0;height:100%;width:{rev_pct:.1f}%;background:{soft};border-radius:4px;"></div>'
                f'<div style="position:absolute;left:0;top:0;height:100%;width:{prof_pct:.1f}%;background:{accent};border-radius:4px;"></div>'
                f'</div>'
                f'<div style="text-align:right;font-size:11.5px;font-variant-numeric:tabular-nums;white-space:nowrap;">'
                f'<b style="color:#0f172a;">${prof/1e3:.1f}k</b> <span style="color:#475569;">({margin:.1f}%)</span> '
                f'<span title="Return rate: {ret:.2f}% | Click-to-order: {conv:.1f}%" '
                f'style="background:{ret_bg};color:{ret_fg};padding:1px 5px;border-radius:4px;font-size:10px;font-weight:700;">{ret:.1f}% ret</span>'
                f'</div></div>'
            )
        sections.append(
            f'<div style="margin-bottom:8px;">'
            f'<div style="font-size:11px;font-weight:700;text-transform:uppercase;color:{accent};letter-spacing:0.4px;margin-bottom:3px;">'
            f'{dept} Top 3 (Solid = Gross Profit | Shaded = Net Revenue)</div>'
            + "".join(rows)
            + "</div>"
        )
    return takeaway, "".join(sections)


def _viz_q2(df: pd.DataFrame):
    max_ev = max((_vf(r.get("latest_14d_events")) for _, r in df.iterrows()), default=1.0) or 1.0
    top_r = df.iloc[0]
    low_r = df.iloc[-1]
    spread = _vf(top_r.get("latest_14d_events")) / max(_vf(low_r.get("latest_14d_events")), 1.0)
    avg_p2p = df["product_to_purchase_pct"].astype(float).mean()
    takeaway = (
        f"<b>{html.escape(_vs(top_r['traffic_source']))}</b> drives <b>{spread:.1f}x</b> more 14d traffic than "
        f"<b>{html.escape(_vs(low_r['traffic_source']))}</b> ({_vi(top_r['latest_14d_events']):,} vs. {_vi(low_r['latest_14d_events']):,}), "
        f"yet view→purchase efficiency stays tightly locked around <b>{avg_p2p:.2f}%</b> across all channels."
    )

    rows = []
    for _, r in df.iterrows():
        ch = _vs(r.get("traffic_source"))
        ev = _vi(r.get("latest_14d_events"))
        buys = _vi(r.get("latest_14d_purchases"))
        v2c = _vf(r.get("view_to_cart_pct"))
        c2p = _vf(r.get("cart_to_purchase_pct"))
        p2p = _vf(r.get("product_to_purchase_pct"))
        ev_w = ev * 100.0 / max_ev
        buy_w = buys * 100.0 / max_ev
        rows.append(
            f'<div style="display:grid;grid-template-columns:76px 1fr 165px;align-items:center;gap:10px;padding:5px 0;border-bottom:1px dashed #e2e8f0;">'
            f'<div style="font-weight:700;font-size:12px;color:#0f172a;">{html.escape(ch)}</div>'
            f'<div>'
            f'<div style="position:relative;height:14px;background:#f1f5f9;border-radius:4px;overflow:hidden;">'
            f'<div style="position:absolute;left:0;top:0;height:100%;width:{ev_w:.1f}%;background:#93c5fd;"></div>'
            f'<div style="position:absolute;left:0;top:0;height:100%;width:{max(buy_w, 1.5):.1f}%;background:#1d4ed8;"></div>'
            f'</div>'
            f'<div style="font-size:10.5px;color:#475569;margin-top:2px;font-variant-numeric:tabular-nums;">'
            f'<b>{ev:,}</b> 14d events → <b style="color:#1d4ed8;">{buys:,}</b> purchases</div>'
            f'</div>'
            f'<div style="background:#f8fafc;border:1px solid #e2e8f0;border-radius:6px;padding:4px 7px;font-size:10.5px;font-variant-numeric:tabular-nums;">'
            f'<div style="display:flex;justify-content:space-between;color:#334155;">'
            f'<span>View→Cart <b>{v2c:.1f}%</b></span><span>Cart→Buy <b>{c2p:.1f}%</b></span></div>'
            f'<div style="margin-top:3px;height:5px;background:#e2e8f0;border-radius:999px;overflow:hidden;display:flex;">'
            f'<div style="width:{v2c:.1f}%;background:#60a5fa;"></div>'
            f'</div>'
            f'<div style="text-align:right;font-weight:700;color:#1967d2;margin-top:2px;">End-to-End: {p2p:.2f}%</div>'
            f'</div></div>'
        )
    return takeaway, "".join(rows)


def _viz_q3(df: pd.DataFrame):
    sub = df.head(7)
    max_val = max(
        max((_vf(r.get("avg_30d_capital_tied_usd")) for _, r in sub.iterrows()), default=1.0),
        max((_vf(r.get("unrealized_margin_pool_usd")) for _, r in sub.iterrows()), default=1.0),
        1.0,
    )
    upside_rows = sub[sub["unrealized_margin_pool_usd"].astype(float) > sub["avg_30d_capital_tied_usd"].astype(float)]
    hero = upside_rows.iloc[0] if not upside_rows.empty else sub.iloc[0]
    takeaway = (
        f"<b>{html.escape(_vs(sub.iloc[0]['department']) + ' · ' + _vs(sub.iloc[0]['category']))}</b> locks "
        f"<b>${_vf(sub.iloc[0]['avg_30d_capital_tied_usd'])/1e3:,.0f}k</b> in shelf cost, while "
        f"<b>{html.escape(_vs(hero['department']) + ' · ' + _vs(hero['category']))}</b> holds "
        f"<b>${_vf(hero['unrealized_margin_pool_usd'])/1e3:,.0f}k</b> in unrealized margin (~{_vf(hero['monthly_sell_through_pct']):.1f}%/mo turnover)."
    )

    legend = (
        '<div style="display:flex;justify-content:space-between;font-size:11px;color:#475569;margin-bottom:6px;">'
        '<span><span style="display:inline-block;width:9px;height:9px;background:#2563eb;border-radius:2px;margin-right:4px;"></span>30d Working Capital Tied ($)</span>'
        '<span><span style="display:inline-block;width:9px;height:9px;background:#10b981;border-radius:2px;margin-right:4px;"></span>Unrealized Margin Upside ($)</span>'
        '<span>30d Sell-Through</span></div>'
    )
    rows = []
    for _, r in sub.iterrows():
        label = f"{_vs(r.get('department'))[:1]} · {_vs(r.get('category'))}"
        cap = _vf(r.get("avg_30d_capital_tied_usd"))
        mar = _vf(r.get("unrealized_margin_pool_usd"))
        st = _vf(r.get("monthly_sell_through_pct"))
        cap_w = cap * 100.0 / max_val
        mar_w = mar * 100.0 / max_val
        star = ' <span title="Unrealized margin exceeds locked capital!" style="color:#059669;font-weight:800;">★</span>' if mar > cap else ""
        rows.append(
            f'<div style="display:grid;grid-template-columns:138px 1fr 78px;align-items:center;gap:8px;margin:5px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;" '
            f'title="{html.escape(label)}">{html.escape(label)}{star}</div>'
            f'<div style="display:flex;flex-direction:column;gap:2px;">'
            f'<div style="display:flex;align-items:center;gap:5px;">'
            f'<div style="height:7px;width:{cap_w:.1f}%;background:#2563eb;border-radius:3px;"></div>'
            f'<span style="font-size:10px;color:#1e40af;font-variant-numeric:tabular-nums;">${cap/1e3:.0f}k</span></div>'
            f'<div style="display:flex;align-items:center;gap:5px;">'
            f'<div style="height:7px;width:{mar_w:.1f}%;background:#10b981;border-radius:3px;"></div>'
            f'<span style="font-size:10px;color:#047857;font-variant-numeric:tabular-nums;">${mar/1e3:.0f}k</span></div>'
            f'</div>'
            f'<div style="text-align:right;"><span style="background:#f1f5f9;color:#334155;padding:2px 6px;border-radius:999px;font-size:10.5px;font-weight:700;">{st:.2f}%/mo</span></div>'
            f'</div>'
        )
    return takeaway, legend + "".join(rows)


def _viz_q4(df: pd.DataFrame):
    sub = df.head(7)
    max_lat = (max((_vf(r.get("latest_30d_events")) for _, r in sub.iterrows()), default=1.0) or 1.0) * 1.32
    top = sub.iloc[0]
    surge = _vf(top.get("latest_30d_events")) / max(_vf(top.get("avg_30d_event_volume")), 1.0)
    avg_ab = sub["avg_cart_abandonment_pct"].astype(float).mean()
    takeaway = (
        f"Latest 30-day traffic surged <b>{surge:.1f}x above the 5-year rolling baseline</b> "
        f"(led by <b>{html.escape(_vs(top['traffic_source']) + ' · ' + _vs(top['browser']))}</b> at {_vi(top['latest_30d_events']):,} events), "
        f"while cart abandonment persists near <b>{avg_ab:.1f}%</b>."
    )

    legend = (
        '<div style="display:flex;justify-content:space-between;font-size:11px;color:#475569;margin-bottom:5px;">'
        '<span>Channel × Browser</span>'
        '<span>Bar = Latest 30d Events &nbsp;|&nbsp; <b style="color:#0f172a;">|</b> = 5-Yr Avg Baseline</span>'
        '<span>Abandonment</span></div>'
    )
    rows = []
    for _, r in sub.iterrows():
        seg = f"{_vs(r.get('traffic_source'))} · {_vs(r.get('browser'))}"
        avg_v = _vf(r.get("avg_30d_event_volume"))
        lat_v = _vf(r.get("latest_30d_events"))
        ab = _vf(r.get("avg_cart_abandonment_pct"))
        vol = _vf(r.get("conversion_volatility_pts"))
        mult = lat_v / max(avg_v, 1.0)
        lat_pct = min(100.0, lat_v * 100.0 / max_lat)
        avg_pct = min(100.0, avg_v * 100.0 / max_lat)
        rows.append(
            f'<div style="display:grid;grid-template-columns:118px 1fr 104px;align-items:center;gap:8px;margin:6px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">{html.escape(seg)}</div>'
            f'<div title="5-Yr Avg: {avg_v:,.0f} | Latest 30d: {lat_v:,.0f} ({mult:.1f}x) | Volatility: ±{vol:.2f} pts" '
            f'style="position:relative;height:15px;background:#f1f5f9;border-radius:4px;">'
            f'<div style="height:100%;width:{lat_pct:.1f}%;background:linear-gradient(90deg,#60a5fa,#2563eb);border-radius:4px;"></div>'
            f'<div style="position:absolute;left:{avg_pct:.1f}%;top:-2px;bottom:-2px;width:2.5px;background:#0f172a;border-radius:1px;"></div>'
            f'<span style="position:absolute;right:6px;top:0;font-size:10px;font-weight:700;color:#334155;line-height:15px;">{_vi(lat_v):,} ({mult:.1f}x)</span>'
            f'</div>'
            f'<div style="text-align:right;font-variant-numeric:tabular-nums;">'
            f'<span style="background:#fee2e2;color:#b91c1c;padding:2px 6px;border-radius:999px;font-size:10.5px;font-weight:700;">{ab:.1f}% aband</span>'
            f'</div></div>'
        )
    return takeaway, legend + "".join(rows)


def _viz_q5(df: pd.DataFrame):
    sub = df.head(7)
    max_peak = (max((_vf(r.get("all_time_peak_90d_profit_usd")) for _, r in sub.iterrows()), default=1.0) or 1.0) * 1.32
    worst_r = sub.loc[sub["worst_90d_return_spike_pct"].astype(float).idxmax()]
    at_peak = sum(1 for _, r in sub.iterrows() if _vf(r.get("current_90d_profit_usd")) >= 0.97 * _vf(r.get("all_time_peak_90d_profit_usd")))
    takeaway = (
        f"<b>{at_peak}/{len(sub)}</b> leading brands sit at <b>97–100% of their 6-year 90d profit peak</b>, "
        f"but return-stress cubes expose up to <b>{_vf(worst_r['worst_90d_return_spike_pct']):.1f}%</b> 90d return spikes "
        f"(<b>{html.escape(_vs(worst_r['brand']))}</b> vs. {_vf(worst_r['historical_avg_return_pct']):.1f}% avg)."
    )

    header = (
        '<div style="display:grid;grid-template-columns:116px 1fr 150px;gap:8px;font-size:10.5px;color:#475569;font-weight:600;margin-bottom:4px;">'
        '<div>Brand</div><div>Current 90d Profit vs. 6-Yr Peak</div><div style="text-align:right;">Return Rate: Avg ●──● Spike</div></div>'
    )
    rows = []
    for _, r in sub.iterrows():
        brand = _vs(r.get("brand"))
        cur = _vf(r.get("current_90d_profit_usd"))
        peak = max(_vf(r.get("all_time_peak_90d_profit_usd")), 1.0)
        avg_ret = _vf(r.get("historical_avg_return_pct"))
        max_ret = _vf(r.get("worst_90d_return_spike_pct"))
        cur_w = cur * 100.0 / max_peak
        peak_w = peak * 100.0 / max_peak
        x1 = min(136.0, max(6.0, (avg_ret / 35.0) * 136.0))
        x2 = min(136.0, max(x1 + 4.0, (max_ret / 35.0) * 136.0))
        rows.append(
            f'<div style="display:grid;grid-template-columns:116px 1fr 150px;align-items:center;gap:8px;margin:5px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;" title="{html.escape(brand)}">{html.escape(brand)}</div>'
            f'<div title="Current 90d: ${cur:,.0f} | All-Time Peak: ${peak:,.0f} ({cur*100/peak:.0f}%)" style="position:relative;height:14px;background:#f1f5f9;border-radius:4px;">'
            f'<div style="position:absolute;left:0;top:0;height:100%;width:{peak_w:.1f}%;background:#dbeafe;border-radius:4px;"></div>'
            f'<div style="position:absolute;left:0;top:0;height:100%;width:{cur_w:.1f}%;background:#1d4ed8;border-radius:4px;"></div>'
            f'<span style="position:absolute;right:5px;top:0;font-size:10px;font-weight:700;color:#1e293b;line-height:14px;">${cur/1e3:.1f}k ({cur*100/peak:.0f}%)</span>'
            f'</div>'
            f'<div style="display:flex;align-items:center;justify-content:flex-end;gap:4px;">'
            f'<svg width="92" height="16" viewBox="0 0 142 16">'
            f'<rect x="0" y="6" width="142" height="4" rx="2" fill="#f1f5f9"/>'
            f'<line x1="{x1:.1f}" y1="8" x2="{x2:.1f}" y2="8" stroke="#ef4444" stroke-width="2.5"/>'
            f'<circle cx="{x1:.1f}" cy="8" r="3.8" fill="#16a34a"/>'
            f'<circle cx="{x2:.1f}" cy="8" r="4.2" fill="#dc2626"/>'
            f'</svg>'
            f'<span style="font-size:10px;font-variant-numeric:tabular-nums;color:#b91c1c;font-weight:700;width:52px;text-align:right;">{avg_ret:.0f}→{max_ret:.0f}%</span>'
            f'</div></div>'
        )
    return takeaway, header + "".join(rows)


def _viz_q6(df: pd.DataFrame):
    sub = df.head(8)
    worst = sub.iloc[0]
    avg_local = sub["avg_local_hub_km"].astype(float).mean()
    takeaway = (
        f"<b>{html.escape(_short_dc(worst['distribution_center']))}</b> averages <b>{_vf(worst['avg_distance_km']):,.0f} km</b> per shipment "
        f"(<b>{_vf(worst['pct_long_haul_gt_1500km']):.1f}% &gt; 1,500 km</b>, P90 {_vf(worst['p90_distance_km']):,.0f} km) vs. "
        f"<b>{avg_local:.1f} km</b> via local metro micro-hubs."
    )

    max_km = max((_vf(r.get("p90_distance_km")) for _, r in sub.iterrows()), default=4200.0) * 1.05
    w_svg = 250.0
    thresh_x = (1500.0 / max_km) * w_svg
    legend = (
        '<div style="display:flex;justify-content:space-between;font-size:10.5px;color:#475569;margin-bottom:4px;">'
        '<span>Warehouse</span>'
        '<span><b style="color:#16a34a;">●</b> Local (~7 km) &nbsp; <b style="color:#0f766e;">●</b> Avg km &nbsp; <b style="color:#94a3b8;">─┤</b> P90 km &nbsp; <span style="color:#dc2626;">┆1.5k km</span></span>'
        '<span>% Long-Haul</span></div>'
    )
    rows = []
    for _, r in sub.iterrows():
        dc = _short_dc(_vs(r.get("distribution_center")))
        loc_km = _vf(r.get("avg_local_hub_km"))
        avg_km = _vf(r.get("avg_distance_km"))
        p90_km = _vf(r.get("p90_distance_km"))
        lh = _vf(r.get("pct_long_haul_gt_1500km"))
        x_loc = max(4.0, (loc_km / max_km) * w_svg)
        x_avg = (avg_km / max_km) * w_svg
        x_p90 = min(w_svg - 4.0, (p90_km / max_km) * w_svg)
        pill_bg, pill_fg = ("#fee2e2", "#b91c1c") if lh >= 50 else ("#fef3c7", "#b45309")
        rows.append(
            f'<div style="display:grid;grid-template-columns:108px 1fr 68px;align-items:center;gap:6px;margin:4px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;">{html.escape(dc)}</div>'
            f'<svg viewBox="0 0 {w_svg:.0f} 18" style="width:100%;height:18px;overflow:visible;">'
            f'<rect x="0" y="7" width="{w_svg:.0f}" height="4" rx="2" fill="#f1f5f9"/>'
            f'<line x1="{thresh_x:.1f}" y1="1" x2="{thresh_x:.1f}" y2="17" stroke="#fca5a5" stroke-dasharray="2,2" stroke-width="1.2"/>'
            f'<line x1="{x_loc:.1f}" y1="9" x2="{x_p90:.1f}" y2="9" stroke="#94a3b8" stroke-width="2"/>'
            f'<line x1="{x_p90:.1f}" y1="4" x2="{x_p90:.1f}" y2="14" stroke="#64748b" stroke-width="1.8"/>'
            f'<circle cx="{x_loc:.1f}" cy="9" r="3.5" fill="#16a34a"><title>Local Micro-Hub: {loc_km:.1f} km</title></circle>'
            f'<circle cx="{x_avg:.1f}" cy="9" r="4.5" fill="#0f766e"><title>Avg Haul: {avg_km:,.1f} km | P90: {p90_km:,.1f} km</title></circle>'
            f'<text x="{min(x_avg + 7, w_svg - 48):.1f}" y="12.5" font-size="9.5" font-weight="700" fill="#0f172a">{avg_km:,.0f} km</text>'
            f'</svg>'
            f'<div style="text-align:right;"><span style="background:{pill_bg};color:{pill_fg};padding:2px 6px;border-radius:999px;font-size:10.5px;font-weight:700;">{lh:.0f}% &gt;1.5k</span></div>'
            f'</div>'
        )
    return takeaway, legend + "".join(rows)


def _viz_q7(df: pd.DataFrame):
    avg_act = df["avg_actual_km"].astype(float).mean()
    avg_ndc = df["avg_nearest_dc_km"].astype(float).mean()
    avg_mh = df["avg_microhub_km"].astype(float).mean()
    tot_saved_m = df["reducible_freight_km"].astype(float).sum() / 1e6
    cut_dc = 100.0 * (1.0 - avg_ndc / max(avg_act, 1.0))
    cut_mh = 100.0 * (1.0 - avg_mh / max(avg_act, 1.0))
    takeaway = (
        f"Routing from the nearest existing warehouse cuts average haul by <b>-{cut_dc:.0f}%</b> "
        f"(<b>{avg_act:,.0f} → {avg_ndc:,.0f} km</b>), saving <b>{tot_saved_m:.1f}M freight-km</b> across top 8 categories "
        f"(micro-hubs reach <b>{avg_mh:.0f} km</b>, <b>-{cut_mh:.0f}%</b>)."
    )

    tier_strip = (
        f'<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;margin-bottom:8px;">'
        f'<div style="background:#fef2f2;border:1px solid #fecaca;border-radius:6px;padding:6px 8px;">'
        f'<div style="font-size:10px;color:#991b1b;font-weight:700;">CURRENT HAUL</div>'
        f'<div style="font-size:15px;font-weight:800;color:#7f1d1d;">{avg_act:,.0f} km</div></div>'
        f'<div style="background:#eff6ff;border:1px solid #bfdbfe;border-radius:6px;padding:6px 8px;">'
        f'<div style="font-size:10px;color:#1d4ed8;font-weight:700;">NEAREST DC (-{cut_dc:.0f}%)</div>'
        f'<div style="font-size:15px;font-weight:800;color:#1e3a8a;">{avg_ndc:,.0f} km</div></div>'
        f'<div style="background:#f0fdf4;border:1px solid #bbf7d0;border-radius:6px;padding:6px 8px;">'
        f'<div style="font-size:10px;color:#15803d;font-weight:700;">MICRO-HUB (-{cut_mh:.0f}%)</div>'
        f'<div style="font-size:15px;font-weight:800;color:#14532d;">{avg_mh:,.0f} km</div></div>'
        f'</div>'
    )
    max_save = max((_vf(r.get("reducible_freight_km")) for _, r in df.iterrows()), default=1.0) or 1.0
    bars = []
    for _, r in df.head(6).iterrows():
        cat = _vs(r.get("category"))
        saved = _vf(r.get("reducible_freight_km"))
        w = saved * 100.0 / max_save
        bars.append(
            f'<div style="display:grid;grid-template-columns:148px 1fr 72px;align-items:center;gap:8px;margin:4px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;" title="{html.escape(cat)}">{html.escape(cat)}</div>'
            f'<div style="height:11px;background:#f1f5f9;border-radius:999px;overflow:hidden;">'
            f'<div style="height:100%;width:{w:.1f}%;background:linear-gradient(90deg,#34d399,#059669);border-radius:999px;"></div></div>'
            f'<div style="text-align:right;font-size:11px;font-weight:700;color:#065f46;font-variant-numeric:tabular-nums;">{saved/1e6:.2f}M km</div>'
            f'</div>'
        )
    return takeaway, tier_strip + "".join(bars)


def _viz_q8(df: pd.DataFrame):
    top = df.iloc[0]
    max_hull_r = df.loc[df["avg_catchment_hull_sq_km"].astype(float).idxmax()]
    takeaway = (
        f"<b>{html.escape(_short_dc(top['distribution_center']))}</b> leads 300 km 180d catchment revenue at "
        f"<b>${_vf(top['current_180d_catchment_rev'])/1e3:.1f}k</b> ({_vi(top['current_180d_buyers_300km'])} buyers), while "
        f"<b>{html.escape(_short_dc(max_hull_r['distribution_center']))}</b> spans the widest convex hull "
        f"(<b>{_vi(max_hull_r['avg_catchment_hull_sq_km']):,} sq km</b>)."
    )

    w, h = 440.0, 215.0
    max_rev = max((_vf(r.get("current_180d_catchment_rev")) for _, r in df.iterrows()), default=1.0) or 1.0
    max_hull = max((_vf(r.get("avg_catchment_hull_sq_km")) for _, r in df.iterrows()), default=1.0) or 1.0

    # Custom label offsets to keep densely clustered East Coast / Gulf hubs crisp and inside map bounds
    lbl_offsets = {
        "NY/NJ Port": (-10, -8, "end"),
        "Philadelphia PA": (-10, 11, "end"),
        "Charleston SC": (-10, -4, "end"),
        "Savannah GA": (-10, 11, "end"),
        "Chicago IL": (0, -13, "middle"),
        "Los Angeles CA": (11, 4, "start"),
        "Houston TX": (-9, 13, "end"),
        "New Orleans LA": (0, 15, "middle"),
        "Mobile AL": (9, 12, "start"),
        "Memphis TN": (-9, -8, "end"),
    }

    nodes_svg = []
    for idx, (_, r) in enumerate(df.iterrows()):
        raw_dc = _vs(r.get("distribution_center"))
        dc = _short_dc(raw_dc)
        pt = _parse_wkt_point(_vs(r.get("current_demand_centroid_wkt"))) or DC_COORDS.get(raw_dc) or (-98.0, 38.0)
        x, y = _us_project(pt[0], pt[1], w, h)
        rev = _vf(r.get("current_180d_catchment_rev"))
        hull = _vf(r.get("avg_catchment_hull_sq_km"))
        buyers = _vi(r.get("current_180d_buyers_300km"))
        r_halo = 9.0 + 14.0 * math.sqrt(max(hull, 0.0) / max_hull)
        r_core = 4.0 + 7.5 * math.sqrt(max(rev, 0.0) / max_rev)
        dx, dy, anchor = lbl_offsets.get(dc, (8, 4, "start"))
        show_lbl = idx < 5 or dc in ("Chicago IL", "Houston TX", "Los Angeles CA")
        lbl_svg = (
            f'<text x="{x + dx:.1f}" y="{y + dy:.1f}" text-anchor="{anchor}" '
            f'font-size="9.5" font-weight="700" fill="#0f172a" '
            f'paint-order="stroke" stroke="#ffffff" stroke-width="2.5">{html.escape(dc)} (${rev/1e3:.0f}k)</text>'
            if show_lbl
            else ""
        )
        nodes_svg.append(
            f'<g><title>{html.escape(dc)} — 180d Rev: ${rev:,.0f} | Buyers (300km): {buyers:,} | Hull: {hull:,.0f} sq km</title>'
            f'<circle cx="{x}" cy="{y}" r="{r_halo:.1f}" fill="#10b981" fill-opacity="0.18" stroke="#059669" stroke-width="1" stroke-dasharray="2,2"/>'
            f'<circle cx="{x}" cy="{y}" r="{r_core:.1f}" fill="#059669" stroke="#ffffff" stroke-width="1.3"/>'
            f'{lbl_svg}</g>'
        )

    svg = (
        f'<svg viewBox="0 0 {w:.0f} {h:.0f}" style="width:100%;height:auto;display:block;">'
        + _us_base_svg_elements(w, h)
        + "".join(nodes_svg)
        + f'<text x="10" y="{h - 7:.0f}" font-size="8.8" fill="#475569">'
        f'Dashed halo = 300 km Convex Hull area (sq km) • Solid bubble = 180d Catchment Revenue (WKT centroids)</text>'
        + "</svg>"
    )
    return takeaway, svg


def _viz_q9(df: pd.DataFrame):
    men = df[df["department"] == "Men"].sort_values("forecast_week")
    women = df[df["department"] == "Women"].sort_values("forecast_week")
    m_tot = men["predicted_revenue"].astype(float).sum()
    w_tot = women["predicted_revenue"].astype(float).sum()
    takeaway = (
        f"Zero-shot <code>TimesFM 2.5</code> projects <b>${m_tot/1e3:.1f}k</b> 4-week revenue for <b>Men</b> "
        f"and <b>${w_tot/1e3:.1f}k</b> for <b>Women</b>, with calibrated 90% confidence ribbons."
    )

    w, h = 440.0, 195.0
    pad_l, pad_r, pad_t, pad_b = 44.0, 20.0, 22.0, 28.0
    lo_min = min((_vf(r.get("lower_bound_90")) for _, r in df.iterrows()), default=30000.0) * 0.92
    hi_max = max((_vf(r.get("upper_bound_90")) for _, r in df.iterrows()), default=85000.0) * 1.05
    span = max(hi_max - lo_min, 1.0)

    def _xy(idx: int, n: int, val: float):
        x = pad_l + (idx / max(n - 1, 1)) * (w - pad_l - pad_r)
        y = pad_t + (1.0 - (val - lo_min) / span) * (h - pad_t - pad_b)
        return round(x, 1), round(y, 1)

    grids = []
    for step in range(4):
        gv = lo_min + (step / 3.0) * span
        _, gy = _xy(0, 4, gv)
        grids.append(
            f'<line x1="{pad_l}" y1="{gy}" x2="{w - pad_r}" y2="{gy}" stroke="#e2e8f0" stroke-dasharray="3,3"/>'
            f'<text x="{pad_l - 6}" y="{gy + 3}" text-anchor="end" font-size="9.5" fill="#64748b">${gv/1e3:.0f}k</text>'
        )

    series_svg = []
    x_labels = []
    for s_df, color, label in ((men, "#1967d2", "Men"), (women, "#9333ea", "Women")):
        if s_df.empty:
            continue
        n = len(s_df)
        upper_pts, lower_pts, mid_pts, dots = [], [], [], []
        for i, (_, r) in enumerate(s_df.iterrows()):
            wk = _vs(r.get("forecast_week"))[5:]
            pred = _vf(r.get("predicted_revenue"))
            lo = _vf(r.get("lower_bound_90"))
            hi = _vf(r.get("upper_bound_90"))
            ux, uy = _xy(i, n, hi)
            lx, ly = _xy(i, n, lo)
            mx, my = _xy(i, n, pred)
            upper_pts.append(f"{ux},{uy}")
            lower_pts.append(f"{lx},{ly}")
            mid_pts.append(f"{mx},{my}")
            lbl_dy = -8 if label == "Men" else 13
            dots.append(
                f'<circle cx="{mx}" cy="{my}" r="3.8" fill="{color}" stroke="#fff" stroke-width="1.2">'
                f'<title>{label} ({wk}): ${pred:,.0f} [90% CI: ${lo:,.0f} – ${hi:,.0f}]</title></circle>'
                f'<text x="{mx}" y="{my + lbl_dy}" text-anchor="middle" font-size="9.5" font-weight="700" fill="{color}">${pred/1e3:.1f}k</text>'
            )
            if label == "Men":
                x_labels.append(f'<text x="{mx}" y="{h - 8}" text-anchor="middle" font-size="9.5" fill="#475569">{html.escape(wk)}</text>')
        poly = " ".join(upper_pts + list(reversed(lower_pts)))
        series_svg.append(
            f'<polygon points="{poly}" fill="{color}" fill-opacity="0.13"/>'
            f'<polyline points="{" ".join(mid_pts)}" fill="none" stroke="{color}" stroke-width="2.4"/>'
            + "".join(dots)
        )

    legend_svg = (
        f'<g transform="translate({pad_l}, 12)">'
        f'<rect x="0" y="-7" width="10" height="8" rx="2" fill="#1967d2"/>'
        f'<text x="14" y="0" font-size="10" font-weight="700" fill="#1967d2">Men (90% CI)</text>'
        f'<rect x="98" y="-7" width="10" height="8" rx="2" fill="#9333ea"/>'
        f'<text x="112" y="0" font-size="10" font-weight="700" fill="#9333ea">Women (90% CI)</text>'
        f'</g>'
    )
    svg = f'<svg viewBox="0 0 {w:.0f} {h:.0f}" style="width:100%;height:auto;display:block;">' + "".join(grids) + "".join(series_svg) + "".join(x_labels) + legend_svg + "</svg>"
    return takeaway, svg


def _viz_q10(df: pd.DataFrame):
    sub = df.head(7)
    top = sub.iloc[0]
    surge_pct = (_vf(top["actual_items"]) / max(_vf(top["expected_max"]), 1.0) - 1.0) * 100.0
    takeaway = (
        f"Flagged <b>{len(df)}</b> recent 60-day volume breakouts (p &ge; {_vf(df['anomaly_prob'].astype(float).min()):.3f}), peaking on "
        f"<b>{html.escape(_vs(top['order_date']))} ({html.escape(_vs(top['department']))})</b> at "
        f"<b>{_vi(top['actual_items'])} items</b> (+{surge_pct:.0f}% above normal ceiling)."
    )

    max_x = max((_vf(r.get("actual_items")) for _, r in sub.iterrows()), default=180.0) * 1.18
    w_svg = 235.0
    legend = (
        '<div style="display:flex;justify-content:space-between;font-size:10.5px;color:#475569;margin-bottom:4px;">'
        '<span>Date · Dept</span>'
        '<span><span style="display:inline-block;width:14px;height:6px;background:#cbd5e1;border-radius:2px;"></span> Expected Normal Band &nbsp; <b style="color:#dc2626;">●</b> Actual Spike</span>'
        '<span>Breakout</span></div>'
    )
    rows = []
    for _, r in sub.iterrows():
        lbl = f"{_vs(r.get('order_date'))[5:]} · {_vs(r.get('department'))[:1]}"
        lo = _vf(r.get("expected_min"))
        hi = _vf(r.get("expected_max"))
        act = _vf(r.get("actual_items"))
        prob = _vf(r.get("anomaly_prob")) * 100.0
        x_lo = (lo / max_x) * w_svg
        x_hi = (hi / max_x) * w_svg
        x_act = (act / max_x) * w_svg
        over = act - hi
        rows.append(
            f'<div style="display:grid;grid-template-columns:86px 1fr 94px;align-items:center;gap:8px;margin:5px 0;">'
            f'<div style="font-size:11.5px;font-weight:600;color:#0f172a;font-variant-numeric:tabular-nums;">{html.escape(lbl)}</div>'
            f'<svg viewBox="0 0 {w_svg:.0f} 16" style="width:100%;height:16px;overflow:visible;">'
            f'<rect x="0" y="6" width="{w_svg:.0f}" height="4" rx="2" fill="#f1f5f9"/>'
            f'<rect x="{x_lo:.1f}" y="3" width="{max(x_hi - x_lo, 3.0):.1f}" height="10" rx="3" fill="#cbd5e1">'
            f'<title>Expected Band: {lo:.1f} – {hi:.1f} items</title></rect>'
            f'<line x1="{x_hi:.1f}" y1="8" x2="{x_act:.1f}" y2="8" stroke="#dc2626" stroke-width="2" stroke-dasharray="2,2"/>'
            f'<circle cx="{x_act:.1f}" cy="8" r="4.5" fill="#dc2626"><title>Actual: {act:.0f} items (p={prob:.2f}%)</title></circle>'
            f'<text x="{x_act + 7:.1f}" y="11.5" font-size="9.5" font-weight="700" fill="#991b1b">{_vi(act)}</text>'
            f'</svg>'
            f'<div style="text-align:right;font-variant-numeric:tabular-nums;">'
            f'<span style="background:#fee2e2;color:#b91c1c;padding:2px 6px;border-radius:999px;font-size:10px;font-weight:700;">+{over:.0f} ({prob:.1f}%)</span>'
            f'</div></div>'
        )
    return takeaway, legend + "".join(rows)


def _viz_q11(df: pd.DataFrame):
    all_row = df[df["segment_driver"] == "all"]
    non_all = df[df["segment_driver"] != "all"]
    fastest = non_all.loc[non_all["yoy_growth_pct"].astype(float).idxmax()] if not non_all.empty else df.iloc[0]
    tot_delta = _vf(all_row.iloc[0]["yoy_delta"]) if not all_row.empty else _vf(df.iloc[0]["yoy_delta"])
    takeaway = (
        f"Trailing 6-month revenue expanded by <b>+${tot_delta/1e3:,.0f}k</b> YoY, led in fastest growth by "
        f"<code>{html.escape(_vs(fastest['segment_driver']))}</code> "
        f"(<b>+{_vf(fastest['yoy_growth_pct']):.1f}%</b> across {_vf(fastest['population_share_pct']):.1f}% of buyers)."
    )

    max_h26 = max((_vf(r.get("current_6m_revenue", r.get("h1_2026_revenue"))) for _, r in df.iterrows()), default=1.0) or 1.0
    rows = []
    for _, r in df.head(7).iterrows():
        seg = _vs(r.get("segment_driver"))
        seg_clean = "All Segments (Total)" if seg == "all" else seg.replace("traffic_source=", "").replace("department=", "Dept: ").replace("browse_tier=", "")
        r25 = _vf(r.get("prior_6m_revenue", r.get("h1_2025_revenue")))
        delta = max(_vf(r.get("yoy_delta")), 0.0)
        gpct = _vf(r.get("yoy_growth_pct"))
        pop = _vf(r.get("population_share_pct"))
        w25 = r25 * 100.0 / max_h26
        wd = delta * 100.0 / max_h26
        rows.append(
            f'<div style="display:grid;grid-template-columns:145px 1fr 112px;align-items:center;gap:8px;margin:5px 0;">'
            f'<div style="font-size:11px;font-weight:600;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;" '
            f'title="{html.escape(seg)} ({pop:.1f}% share)">{html.escape(seg_clean)}</div>'
            f'<div title="Prior 6M: ${r25:,.0f} → Current 6M: ${r25+delta:,.0f} (+${delta:,.0f})" '
            f'style="display:flex;height:13px;background:#f1f5f9;border-radius:4px;overflow:hidden;">'
            f'<div style="width:{w25:.1f}%;background:#ddd6fe;"></div>'
            f'<div style="width:{wd:.1f}%;background:#7c3aed;"></div>'
            f'</div>'
            f'<div style="text-align:right;font-size:10.5px;font-variant-numeric:tabular-nums;white-space:nowrap;">'
            f'<b style="color:#6d28d9;">+${delta/1e3:.0f}k</b> <span style="color:#475569;">(+{gpct:.0f}%)</span>'
            f'</div></div>'
        )
    legend = (
        '<div style="font-size:10.5px;color:#475569;margin-bottom:4px;">'
        '<span style="display:inline-block;width:9px;height:9px;background:#ddd6fe;border-radius:2px;margin-right:4px;"></span>Prior 6M Base &nbsp; '
        '<span style="display:inline-block;width:9px;height:9px;background:#7c3aed;border-radius:2px;margin-right:4px;"></span>+YoY Expansion Delta</div>'
    )
    return takeaway, legend + "".join(rows)


def _viz_q12(df: pd.DataFrame):
    seg_colors = {
        "Alpine & Heavy Winter": ("#dbeafe", "#1d4ed8", "#2563eb"),
        "Urban Tailored Coat": ("#f3e8fd", "#6d28d9", "#7c3aed"),
        "Rain & Wind Shell": ("#ccfbf1", "#0f766e", "#0d9488"),
        "Fleece & Mid-Layer": ("#fef3c7", "#b45309", "#d97706"),
    }
    grp = (
        df.groupby("campaign_segment", dropna=False)["realized_revenue"]
        .agg(["count", "sum"])
        .reset_index()
        .sort_values("sum", ascending=False)
    )
    tot_rev = max(float(grp["sum"].sum()), 1.0)
    top_seg = grp.iloc[0]
    takeaway = (
        f"<code>AI.IF</code> verified <b>{len(df)}</b> true winter/weather SKUs (${tot_rev/1e3:.1f}k revenue); "
        f"<code>AI.CLASSIFY</code> ranks <b>{html.escape(_vs(top_seg['campaign_segment']))}</b> #1 "
        f"(${_vf(top_seg['sum'])/1e3:.1f}k across {int(top_seg['count'])} SKUs)."
    )

    segs_bar, chips = [], []
    for _, g in grp.iterrows():
        sname = _vs(g["campaign_segment"], "Other")
        srev = _vf(g["sum"])
        scnt = int(g["count"])
        pct = srev * 100.0 / tot_rev
        bg, fg, solid = seg_colors.get(sname, ("#f1f5f9", "#334155", "#64748b"))
        segs_bar.append(f'<div title="{html.escape(sname)}: ${srev:,.0f} ({pct:.1f}%)" style="width:{pct:.1f}%;background:{solid};height:100%;"></div>')
        chips.append(
            f'<span style="background:{bg};color:{fg};padding:2px 7px;border-radius:999px;font-size:10px;font-weight:700;">'
            f'{html.escape(sname)}: ${srev/1e3:.1f}k ({scnt})</span>'
        )

    sku_rows = []
    max_sku = max((_vf(r.get("realized_revenue")) for _, r in df.head(5).iterrows()), default=1.0) or 1.0
    for _, r in df.head(5).iterrows():
        sname = _vs(r.get("campaign_segment"), "Other")
        bg, fg, solid = seg_colors.get(sname, ("#f1f5f9", "#334155", "#64748b"))
        rev = _vf(r.get("realized_revenue"))
        pname = _vs(r.get("product_name"))
        sku_rows.append(
            f'<div style="display:grid;grid-template-columns:1fr 96px 62px;align-items:center;gap:6px;margin:4px 0;">'
            f'<div style="font-size:11px;color:#0f172a;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;" title="{html.escape(pname)}">'
            f'<span style="display:inline-block;width:7px;height:7px;border-radius:50%;background:{solid};margin-right:5px;"></span>'
            f'<b>{html.escape(_vs(r.get("brand")))}</b> — {html.escape(pname)}</div>'
            f'<div style="height:8px;background:#f1f5f9;border-radius:999px;overflow:hidden;">'
            f'<div style="width:{rev*100/max_sku:.1f}%;height:100%;background:{solid};"></div></div>'
            f'<div style="text-align:right;font-size:11px;font-weight:700;color:#0f172a;font-variant-numeric:tabular-nums;">${rev:,.0f}</div>'
            f'</div>'
        )

    body = (
        f'<div style="height:12px;border-radius:6px;overflow:hidden;display:flex;gap:1px;margin-bottom:6px;">{"".join(segs_bar)}</div>'
        f'<div style="display:flex;flex-wrap:wrap;gap:4px;margin-bottom:8px;">{"".join(chips)}</div>'
        + "".join(sku_rows)
    )
    return takeaway, body


def _viz_q13(df: pd.DataFrame):
    top = df.iloc[0]
    avg_gap = (df["avg_ship_km"].astype(float) - df["potential_microhub_km"].astype(float)).mean()
    takeaway = (
        f"Gemini triages all <b>{len(df)}</b> hubs by combining revenue, returns, and a "
        f"<b>{avg_gap:,.0f} km average micro-hub distance savings</b> (peaking at {_vi(top['avg_ship_km']):,} → {_vi(top['potential_microhub_km'])} km in {html.escape(_short_dc(top['distribution_center']))})."
    )

    max_km = max((_vf(r.get("avg_ship_km")) for _, r in df.iterrows()), default=2500.0) or 2500.0

    def _card_row(r):
        dc = _short_dc(_vs(r.get("distribution_center")))
        tier = _vs(r.get("priority_tier"), "Tier 1")
        ship = _vf(r.get("avg_ship_km"))
        mhub = _vf(r.get("potential_microhub_km"))
        ret = _vf(r.get("return_rate_pct"))
        rev = _vf(r.get("net_revenue_usd"))
        act = _vs(r.get("recommended_action"))
        t_bg, t_fg = ("#fee2e2", "#b91c1c") if ("1" in tier or "0" in tier or "High" in tier) else ("#fef3c7", "#b45309")
        return (
            f'<div style="border:1px solid #e2e8f0;border-radius:7px;padding:6px 9px;margin:5px 0;background:#fafafa;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;gap:6px;">'
            f'<div style="font-size:11.5px;font-weight:700;color:#0f172a;">'
            f'<span style="background:{t_bg};color:{t_fg};padding:1px 6px;border-radius:4px;font-size:10px;margin-right:5px;">{html.escape(tier)}</span>'
            f'{html.escape(dc)} <span style="font-weight:500;color:#64748b;font-size:10.5px;">(${rev/1e3:.0f}k rev · {ret:.1f}% ret)</span></div>'
            f'<div style="font-size:10.5px;font-weight:700;color:#0f766e;white-space:nowrap;">{ship:,.0f} → {mhub:.0f} km</div></div>'
            f'<div style="height:5px;background:#e2e8f0;border-radius:999px;margin:4px 0;position:relative;overflow:hidden;">'
            f'<div style="width:{ship*100/max_km:.1f}%;height:100%;background:#f97316;"></div></div>'
            f'<div style="font-size:11px;color:#334155;line-height:1.35;">✨ {html.escape(act)}</div>'
            f'</div>'
        )

    top_html = "".join(_card_row(r) for _, r in df.head(3).iterrows())
    rest_html = ""
    if len(df) > 3:
        rest_cards = "".join(_card_row(r) for _, r in df.iloc[3:].iterrows())
        rest_html = (
            f'<details style="margin-top:4px;"><summary style="cursor:pointer;font-size:11px;color:#7627bb;font-weight:600;">'
            f'+ Show remaining {len(df) - 3} warehouse AI action briefs</summary>{rest_cards}</details>'
        )
    return takeaway, top_html + rest_html


def _viz_q14(df: pd.DataFrame):
    sub = df.head(8)
    tot_trans = max(int(df["cross_order_transitions"].astype(int).sum()), 1)
    tot_sw = int(df["cross_hub_switches"].astype(int).sum())
    sw_pct = tot_sw * 100.0 / tot_trans
    top_dest = df.groupby("repeat_order_category")["downstream_repeat_revenue"].sum().sort_values(ascending=False)
    magnet_cat = _vs(top_dest.index[0]) if len(top_dest) else _vs(sub.iloc[0]["repeat_order_category"])
    magnet_rev = float(top_dest.iloc[0]) if len(top_dest) else 0.0
    takeaway = (
        f"Across <b>8 graph hops</b>, <b>{html.escape(magnet_cat)}</b> is the #1 repeat-purchase magnet "
        f"(<b>${magnet_rev/1e3:.1f}k</b> downstream inflow), and <b>{sw_pct:.1f}%</b> of repeat orders switch fulfillment hubs."
    )

    w, h = 440.0, 215.0
    left_cats = list(dict.fromkeys(_vs(r["first_order_category"]) for _, r in sub.iterrows()))
    right_cats = list(dict.fromkeys(_vs(r["repeat_order_category"]) for _, r in sub.iterrows()))
    lx, rx = 128.0, 312.0

    def _node_y(idx: int, total: int) -> float:
        if total <= 1:
            return h / 2.0
        return 22.0 + idx * ((h - 44.0) / (total - 1))

    l_pos = {c: _node_y(i, len(left_cats)) for i, c in enumerate(left_cats)}
    r_pos = {c: _node_y(i, len(right_cats)) for i, c in enumerate(right_cats)}
    max_t = max((_vf(r.get("cross_order_transitions")) for _, r in sub.iterrows()), default=1.0) or 1.0
    max_rev = max((_vf(r.get("downstream_repeat_revenue")) for _, r in sub.iterrows()), default=1.0) or 1.0

    ribbons = []
    for _, r in sub.iterrows():
        c1 = _vs(r.get("first_order_category"))
        c2 = _vs(r.get("repeat_order_category"))
        tr = _vf(r.get("cross_order_transitions"))
        rev = _vf(r.get("downstream_repeat_revenue"))
        sw = _vi(r.get("cross_hub_switches"))
        days = _vf(r.get("avg_days_to_next_order"))
        y1, y2 = l_pos[c1], r_pos[c2]
        sw_w = 2.5 + 8.5 * (tr / max_t)
        color = "#d97706" if rev >= 0.7 * max_rev else "#f59e0b"
        ribbons.append(
            f'<path d="M {lx + 6},{y1:.1f} C {(lx + rx)/2:.1f},{y1:.1f} {(lx + rx)/2:.1f},{y2:.1f} {rx - 6},{y2:.1f}" '
            f'fill="none" stroke="{color}" stroke-opacity="0.48" stroke-width="{sw_w:.1f}">'
            f'<title>{html.escape(c1)} → {html.escape(c2)}: {int(tr)} transitions | ${rev:,.0f} repeat rev | {sw} hub switches | avg {days:.0f}d</title></path>'
        )

    nodes = []
    for c, y in l_pos.items():
        short_c = c if len(c) <= 20 else c[:19] + "…"
        nodes.append(
            f'<rect x="{lx - 4}" y="{y - 7:.1f}" width="8" height="14" rx="2" fill="#b45309"/>'
            f'<text x="{lx - 10}" y="{y + 3.5:.1f}" text-anchor="end" font-size="10" font-weight="600" fill="#0f172a">{html.escape(short_c)}</text>'
        )
    for c, y in r_pos.items():
        short_c = c if len(c) <= 20 else c[:19] + "…"
        nodes.append(
            f'<rect x="{rx - 4}" y="{y - 7:.1f}" width="8" height="14" rx="2" fill="#0f766e"/>'
            f'<text x="{rx + 10}" y="{y + 3.5:.1f}" text-anchor="start" font-size="10" font-weight="700" fill="#0f766e">{html.escape(short_c)}</text>'
        )

    svg = (
        f'<svg viewBox="0 0 {w:.0f} {h:.0f}" style="width:100%;height:auto;display:block;background:#fffbeb;border:1px solid #fde68a;border-radius:8px;">'
        f'<text x="{lx - 10}" y="12" text-anchor="end" font-size="9.5" font-weight="700" fill="#92400e">1ST ORDER CATEGORY</text>'
        f'<text x="{(lx+rx)/2:.0f}" y="12" text-anchor="middle" font-size="9.5" font-weight="700" fill="#b45309">8-HOP GRAPH BRIDGES</text>'
        f'<text x="{rx + 10}" y="12" text-anchor="start" font-size="9.5" font-weight="700" fill="#0f766e">REPEAT CATEGORY</text>'
        + "".join(ribbons)
        + "".join(nodes)
        + "</svg>"
    )
    return takeaway, svg


def _viz_q15(df: pd.DataFrame):
    tot_margin = df["gross_margin_usd"].astype(float).sum()
    tot_stock = df["hub_unsold_capital_usd"].astype(float).sum()
    avg_h = df["avg_haul_km"].astype(float).mean()
    avg_m = df["avg_microhub_km"].astype(float).mean()
    takeaway = (
        f"Converging <b>Property Graph + 2.42M Clickstream SQL + 490k Stock SQL + Spherical GIS + Gemini</b> across all 7 dataset tables "
        f"isolates <b>{len(df)} long-haul corridors</b> (${tot_margin/1e3:.1f}k margin, ${tot_stock/1e3:,.0f}k unsold origin stock) "
        f"where forward micro-hubs compress shipping distance from <b>{avg_h:,.0f} km → {avg_m:.0f} km</b>."
    )

    w, h = 470.0, 265.0
    max_m = max((_vf(r.get("gross_margin_usd")) for _, r in df.iterrows()), default=1.0) or 1.0
    arcs_svg, pins_svg = [], []
    seen_origins, seen_dests = {}, {}

    for idx, (_, r) in enumerate(df.iterrows()):
        orig = _vs(r.get("origin_hub"))
        fhub = _vs(r.get("top_forward_microhub"))
        dstate = _vs(r.get("dest_state"))
        margin = _vf(r.get("gross_margin_usd"))
        haul = _vf(r.get("avg_haul_km"))
        mhub_km = _vf(r.get("avg_microhub_km"))

        o_lonlat = DC_COORDS.get(orig) or (-90.0, 34.0)
        d_lonlat = MICROHUB_COORDS.get(fhub) or STATE_CENTROIDS.get(dstate) or (-118.2, 34.0)
        x1, y1 = _us_project(o_lonlat[0], o_lonlat[1], w, h)
        x2, y2 = _us_project(d_lonlat[0], d_lonlat[1], w, h)
        cx = (x1 + x2) / 2.0
        cy = min(y1, y2) - (26.0 + idx * 7.5)
        sw = 1.8 + 3.2 * (margin / max_m)
        arcs_svg.append(
            f'<path d="M {x1},{y1} Q {cx:.1f},{cy:.1f} {x2},{y2}" fill="none" stroke="#dc2626" '
            f'stroke-opacity="0.62" stroke-width="{sw:.1f}" stroke-linecap="round">'
            f'<title>{html.escape(_short_dc(orig))} → {html.escape(fhub)}: {haul:,.0f} km → {mhub_km:.0f} km (${margin:,.0f} margin)</title></path>'
        )
        seen_origins[_short_dc(orig)] = (x1, y1)
        seen_dests[fhub] = (x2, y2)

    orig_lbl_offsets = {
        "NY/NJ Port": (-8, -8, "end"),
        "Philadelphia PA": (-8, 12, "end"),
        "Memphis TN": (0, -9, "middle"),
        "Mobile AL": (8, 13, "start"),
        "Houston TX": (-8, 13, "end"),
        "Savannah GA": (8, 5, "start"),
        "Charleston SC": (8, -5, "start"),
    }
    for name, (px, py) in seen_origins.items():
        dx, dy, anc = orig_lbl_offsets.get(name, (7, -8, "start"))
        pins_svg.append(
            f'<circle cx="{px}" cy="{py}" r="4.8" fill="#dc2626" stroke="#fff" stroke-width="1.4"/>'
            f'<text x="{px + dx}" y="{py + dy}" text-anchor="{anc}" font-size="9.5" font-weight="700" fill="#7f1d1d" '
            f'paint-order="stroke" stroke="#ffffff" stroke-width="2.5">{html.escape(name)}</text>'
        )
    dest_lbl_offsets = {
        "San Jose, CA": (9, -6, "start"),
        "Los Angeles, CA": (10, -5, "start"),
        "Torrance, CA": (10, 7, "start"),
        "Chula Vista, CA": (9, 13, "start"),
        "San Francisco, CA": (9, -8, "start"),
        "San Diego, CA": (9, 13, "start"),
    }
    for name, (px, py) in seen_dests.items():
        dx, dy, anc = dest_lbl_offsets.get(name, (8, -5, "start"))
        pins_svg.append(
            f'<circle cx="{px}" cy="{py}" r="5.2" fill="#059669" stroke="#fff" stroke-width="1.4"/>'
            f'<text x="{px + dx}" y="{py + dy}" text-anchor="{anc}" font-size="9.5" font-weight="700" fill="#065f46" '
            f'paint-order="stroke" stroke="#ffffff" stroke-width="2.5">{html.escape(name)}</text>'
        )

    map_svg = (
        f'<svg viewBox="0 0 {w:.0f} {h:.0f}" style="width:100%;height:auto;display:block;">'
        + _us_base_svg_elements(w, h)
        + "".join(arcs_svg)
        + "".join(pins_svg)
        + f'<text x="12" y="{h - 8:.0f}" font-size="9.5" fill="#475569">'
        f'Crimson = Origin Warehouse &amp; Long-Haul Corridor • Emerald = Recommended Forward Micro-Hub</text>'
        + "</svg>"
    )

    cards = []
    for _, r in df.iterrows():
        orig = _short_dc(_vs(r.get("origin_hub")))
        dest = _vs(r.get("dest_state"))
        cat = _vs(r.get("category"))
        margin = _vf(r.get("gross_margin_usd"))
        ret = _vf(r.get("return_rate_pct"))
        clicks = _vf(r.get("avg_buyer_clicks"))
        stock = _vf(r.get("hub_unsold_capital_usd"))
        haul = _vf(r.get("avg_haul_km"))
        mhub_km = _vf(r.get("avg_microhub_km"))
        fhub = _vs(r.get("top_forward_microhub"))
        tier = _vs(r.get("priority_tier"), "P0")
        plan = _vs(r.get("forward_hub_plan"))
        cx = _vs(r.get("cx_retention_play"))
        cards.append(
            f'<div title="CX Play: {html.escape(cx)}" style="border:1px solid #fecaca;background:#fffafa;border-radius:8px;padding:7px 10px;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;gap:6px;margin-bottom:4px;">'
            f'<div style="font-size:11.5px;font-weight:700;color:#0f172a;">'
            f'<span style="background:#dc2626;color:#fff;padding:1px 6px;border-radius:4px;font-size:10px;margin-right:5px;">{html.escape(tier)}</span>'
            f'{html.escape(orig)} → {html.escape(dest)} <span style="color:#64748b;font-weight:600;">({html.escape(cat)})</span></div>'
            f'<div style="font-size:10.5px;font-weight:700;color:#059669;white-space:nowrap;">{haul:,.0f} → {mhub_km:.0f} km</div></div>'
            f'<div style="display:flex;flex-wrap:wrap;gap:4px;margin-bottom:4px;font-size:10px;">'
            f'<span style="background:#fef3c7;color:#92400e;padding:1px 6px;border-radius:4px;font-weight:600;">🕸️ Graph: ${margin:,.0f} margin · {ret:.1f}% ret</span>'
            f'<span style="background:#dbeafe;color:#1e40af;padding:1px 6px;border-radius:4px;font-weight:600;">📊 SQL: {clicks:.1f} clicks · ${stock/1e3:.0f}k stock</span>'
            f'<span style="background:#dcfce7;color:#166534;padding:1px 6px;border-radius:4px;font-weight:600;">🌍 GIS: {html.escape(fhub)}</span>'
            f'</div>'
            f'<div style="font-size:11px;color:#1e293b;line-height:1.35;">✨ {html.escape(plan)}</div>'
            f'</div>'
        )

    hero_html = (
        f'<div style="display:grid;grid-template-columns:minmax(320px, 1.05fr) minmax(340px, 1.25fr);gap:14px;align-items:start;">'
        f'<div>{map_svg}</div>'
        f'<div style="display:grid;grid-template-columns:1fr;gap:6px;">{"".join(cards)}</div>'
        f'</div>'
    )
    return takeaway, hero_html


VIZ_BUILDERS = {
    1: _viz_q1,
    2: _viz_q2,
    3: _viz_q3,
    4: _viz_q4,
    5: _viz_q5,
    6: _viz_q6,
    7: _viz_q7,
    8: _viz_q8,
    9: _viz_q9,
    10: _viz_q10,
    11: _viz_q11,
    12: _viz_q12,
    13: _viz_q13,
    14: _viz_q14,
    15: _viz_q15,
}

ENGINE_CSS_CLASS = {
    "SQL": "eng-sql",
    "GIS": "eng-gis",
    "AI.*": "eng-ai",
    "Graph": "eng-graph",
    "All-in-One": "eng-all4",
}


def render_visual_insights_grid(states: list[dict]) -> None:
    if not states:
        display(HTML('<div style="padding:12px;background:#fef7e0;color:#b06000;border-radius:8px;">Run Step 2 first to populate RESULTS.</div>'))
        return

    cards_html = []
    for s in states:
        qid = int(s.get("id", 0))
        eng = _vs(s.get("engine"))
        eng_cls = ENGINE_CSS_CLASS.get(eng, "eng-sql")
        bg_pill, fg_pill = ENGINE_COLORS.get(eng, ("#f1f5f9", "#334155"))
        span_style = "grid-column: 1 / -1;" if qid == 15 else ""
        df = s.get("df")

        if df is None or (hasattr(df, "empty") and df.empty):
            err_msg = html.escape(_vs(s.get("error"), "No rows returned"))
            takeaway_html = ""
            viz_body = f'<div style="padding:10px;background:#fef2f2;color:#b91c1c;border-radius:6px;font-size:12px;">{err_msg}</div>'
        else:
            builder = VIZ_BUILDERS.get(qid)
            try:
                takeaway_text, viz_body = builder(df) if builder else ("", df.head(5).to_html(index=False, border=0))
                takeaway_html = (
                    f'<div style="background:#f8fafc;border-left:3px solid {fg_pill};padding:6px 10px;'
                    f'border-radius:0 6px 6px 0;font-size:11.5px;color:#334155;line-height:1.4;margin-bottom:10px;">'
                    f'{takeaway_text}</div>'
                )
            except Exception as exc:  # noqa: BLE001
                takeaway_html = ""
                viz_body = f'<div style="padding:8px;color:#b45309;background:#fef3c7;border-radius:6px;font-size:11.5px;">Fallback view ({html.escape(str(exc))})</div>'

        avg_slots = max(round(_vf(s.get("slot_sec")) / max(_vf(s.get("elapsed"), 0.01), 0.01)), 1)
        cards_html.append(f'''
        <div class="bq-viz-card {eng_cls}" style="{span_style}background:#ffffff;border:1px solid #cbd5e1;
                    border-top:4px solid {fg_pill};border-radius:10px;padding:14px 16px;
                    box-shadow:0 1px 3px rgba(15,23,42,0.05);display:flex;flex-direction:column;">
          <div style="display:flex;justify-content:space-between;align-items:flex-start;gap:8px;margin-bottom:8px;">
            <div>
              <div style="display:flex;align-items:center;gap:6px;margin-bottom:3px;">
                <span style="font-weight:800;font-size:12px;color:#0f172a;background:#f1f5f9;padding:2px 7px;border-radius:5px;">Q{qid}</span>
                <span style="padding:2px 8px;border-radius:6px;background:{bg_pill};color:{fg_pill};font-weight:700;font-size:11px;">{html.escape(eng)}</span>
              </div>
              <div style="font-weight:700;font-size:13.5px;color:#0f172a;line-height:1.3;">{html.escape(_vs(s.get("title")))}</div>
            </div>
            <div style="text-align:right;white-space:nowrap;font-size:11px;color:#475569;font-variant-numeric:tabular-nums;">
              <div>⏱ <b>{_vf(s.get("elapsed")):.2f}s</b></div>
              <div>⚡ <b>{_vf(s.get("slot_sec")):,.1f}s</b> (~{avg_slots:,} slots)</div>
            </div>
          </div>
          {takeaway_html}
          <div style="flex:1;">{viz_body}</div>
        </div>
        ''')

    total_slots_s = sum(_vf(s.get("slot_sec")) for s in states)
    grid_ui = f'''
    <style>
      .bq-viz-root {{ font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; color:#0f172a; margin:8px 0; }}
      .bq-tab-radio {{ display:none; }}
      .bq-tab-bar {{ display:flex; flex-wrap:wrap; gap:6px; align-items:center; margin-bottom:14px; background:#f8fafc; padding:8px 10px; border:1px solid #cbd5e1; border-radius:10px; }}
      .bq-tab-lbl {{ cursor:pointer; padding:5px 12px; border-radius:999px; font-size:12px; font-weight:700; color:#475569; background:#ffffff; border-color:#cbd5e1; border:1px solid #cbd5e1; transition:all 0.15s ease; user-select:none; }}
      .bq-tab-lbl:hover {{ border-color:#94a3b8; color:#0f172a; }}
      #bq_tab_all:checked ~ .bq-tab-bar label[for="bq_tab_all"] {{ background:#0f172a; color:#ffffff; border-color:#0f172a; }}
      #bq_tab_sql:checked ~ .bq-tab-bar label[for="bq_tab_sql"] {{ background:#1967d2; color:#ffffff; border-color:#1967d2; }}
      #bq_tab_gis:checked ~ .bq-tab-bar label[for="bq_tab_gis"] {{ background:#137333; color:#ffffff; border-color:#137333; }}
      #bq_tab_ai:checked ~ .bq-tab-bar label[for="bq_tab_ai"] {{ background:#7627bb; color:#ffffff; border-color:#7627bb; }}
      #bq_tab_graph:checked ~ .bq-tab-bar label[for="bq_tab_graph"] {{ background:#b06000; color:#ffffff; border-color:#b06000; }}
      #bq_tab_all4:checked ~ .bq-tab-bar label[for="bq_tab_all4"] {{ background:#c5221f; color:#ffffff; border-color:#c5221f; }}
      #bq_tab_sql:checked ~ .bq-bento-grid > .bq-viz-card:not(.eng-sql) {{ display:none !important; }}
      #bq_tab_gis:checked ~ .bq-bento-grid > .bq-viz-card:not(.eng-gis) {{ display:none !important; }}
      #bq_tab_ai:checked ~ .bq-bento-grid > .bq-viz-card:not(.eng-ai) {{ display:none !important; }}
      #bq_tab_graph:checked ~ .bq-bento-grid > .bq-viz-card:not(.eng-graph) {{ display:none !important; }}
      #bq_tab_all4:checked ~ .bq-bento-grid > .bq-viz-card:not(.eng-all4) {{ display:none !important; }}
      .bq-bento-grid {{ display:grid; grid-template-columns:repeat(auto-fit, minmax(460px, 1fr)); gap:14px; align-items:stretch; }}
    </style>
    <div class="bq-viz-root">
      <input type="radio" name="bq_viz_filter" id="bq_tab_all" class="bq-tab-radio" checked>
      <input type="radio" name="bq_viz_filter" id="bq_tab_sql" class="bq-tab-radio">
      <input type="radio" name="bq_viz_filter" id="bq_tab_gis" class="bq-tab-radio">
      <input type="radio" name="bq_viz_filter" id="bq_tab_ai" class="bq-tab-radio">
      <input type="radio" name="bq_viz_filter" id="bq_tab_graph" class="bq-tab-radio">
      <input type="radio" name="bq_viz_filter" id="bq_tab_all4" class="bq-tab-radio">
      <div class="bq-tab-bar">
        <span style="font-size:12.5px;font-weight:800;color:#0f172a;margin-right:4px;">📊 Visual Insights Grid:</span>
        <label for="bq_tab_all" class="bq-tab-lbl">All 15 Queries</label>
        <label for="bq_tab_sql" class="bq-tab-lbl">SQL (5)</label>
        <label for="bq_tab_gis" class="bq-tab-lbl">GIS (3)</label>
        <label for="bq_tab_ai" class="bq-tab-lbl">AI.* (5)</label>
        <label for="bq_tab_graph" class="bq-tab-lbl">Graph (1)</label>
        <label for="bq_tab_all4" class="bq-tab-lbl">All-in-One (1)</label>
        <span style="margin-left:auto;font-size:11.5px;color:#64748b;">Rendered in-memory from <b>{len(states)}</b> parallel results ({total_slots_s:,.1f} slot-sec)</span>
      </div>
      <div class="bq-bento-grid">
        {"".join(cards_html)}
      </div>
    </div>
    '''
    display(HTML(grid_ui))


print(
    f"✅ Ready — BigQuery client targeting `{TABLES_FQN}` & `{GRAPH_DATASET_FQN}.ecommerce_graph` ({LOCATION}) | Snapshot anchor: {SNAP_END}"
)

---
## Analytics & AI
Run 15 concurrent queries uniting **`SQL`**, **`GIS`**, **`AI.*`** (`TimesFM 2.5` & Gemini), and ISO **`Graph`** (`GQL`) in one serverless engine.

In [ ]:
# @title 1. Define the 15 Queries
QUERIES = [
    {
        "id": 1,
        "engine": "SQL",
        "title": "Full-Funnel Brand Profitability Leaderboard (Views → Stock → Margin)",
        "feature": "4-table join across clickstream events + inventory units + orders + RANK() QUALIFY",
        "sql": rf"""
-- Query 1: Full-Funnel Brand Profitability Leaderboard
-- Purpose: Join clickstream product views, physical inventory items, and orders to rank top 3 brands per department.
WITH product_clicks AS (
  SELECT
    SAFE_CAST(REGEXP_EXTRACT(uri, r'/product/(\d+)') AS INT64) AS product_id,
    COUNT(*) AS product_page_views
  FROM `{TABLES_FQN}.events`
  WHERE event_type = 'product'
  GROUP BY product_id
),
product_stock AS (
  SELECT
    product_id,
    COUNTIF(sold_at IS NULL) AS unsold_units
  FROM `{TABLES_FQN}.inventory_items`
  GROUP BY product_id
),
brand_perf AS (
  SELECT
    p.department,
    p.brand,
    MAX(pc_brand.brand_views) AS clickstream_views,
    COUNT(oi.id) AS items_ordered,
    ROUND(SAFE_DIVIDE(COUNTIF(oi.status = 'Returned') * 100.0, COUNT(oi.id)), 2) AS return_rate_pct,
    ROUND(SUM(IF(oi.status NOT IN ('Cancelled', 'Returned'), oi.sale_price, 0)), 2) AS net_revenue,
    ROUND(SUM(IF(oi.status NOT IN ('Cancelled', 'Returned'), oi.sale_price - p.cost, 0)), 2) AS gross_profit
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  LEFT JOIN (
    SELECT p2.department, p2.brand, SUM(COALESCE(pc.product_page_views, 0)) AS brand_views
    FROM `{TABLES_FQN}.products` AS p2
    LEFT JOIN product_clicks AS pc ON p2.id = pc.product_id
    LEFT JOIN product_stock AS ps ON p2.id = ps.product_id
    GROUP BY p2.department, p2.brand
  ) AS pc_brand
    ON p.department = pc_brand.department AND p.brand = pc_brand.brand
  GROUP BY p.department, p.brand
  HAVING items_ordered >= 100
)
SELECT
  department,
  RANK() OVER (PARTITION BY department ORDER BY gross_profit DESC) AS profit_rank,
  brand,
  clickstream_views,
  items_ordered,
  net_revenue,
  gross_profit,
  ROUND(SAFE_DIVIDE(gross_profit * 100.0, net_revenue), 1) AS gross_margin_pct,
  return_rate_pct
FROM brand_perf
QUALIFY profit_rank <= 3
ORDER BY department, profit_rank;
""",
    },
    {
        "id": 2,
        "engine": "SQL",
        "title": "5-Year Daily Rolling 14-Day Clickstream Stage Funnel",
        "feature": "High-concurrency temporal join: clickstream events × 1,826 rolling 14d regional funnels",
        "sql": rf"""
-- Query 2: 5-Year Daily Rolling 14-Day Clickstream Stage Funnel
-- Purpose: Evaluate 1,826 daily rolling 14-day regional funnels over raw events to measure view-to-cart and cart-to-purchase velocity.
WITH daily_windows AS (
  SELECT report_date
  FROM UNNEST(GENERATE_DATE_ARRAY('{SNAP_5Y_START}', '{SNAP_END}')) AS report_date
),
daily_stage_funnel AS (
  SELECT
    w.report_date,
    e.traffic_source,
    e.state,
    COUNT(*) AS events_14d,
    COUNTIF(e.event_type = 'product') AS product_views_14d,
    COUNTIF(e.event_type = 'cart') AS cart_adds_14d,
    COUNTIF(e.event_type = 'purchase') AS purchases_14d
  FROM `{TABLES_FQN}.events` AS e
  INNER JOIN daily_windows AS w
    ON DATE(e.created_at) BETWEEN DATE_SUB(w.report_date, INTERVAL 13 DAY) AND w.report_date
  GROUP BY w.report_date, e.traffic_source, e.state
),
daily_channel_totals AS (
  SELECT
    report_date,
    traffic_source,
    SUM(events_14d) AS events_14d,
    SUM(product_views_14d) AS product_views_14d,
    SUM(cart_adds_14d) AS cart_adds_14d,
    SUM(purchases_14d) AS purchases_14d
  FROM daily_stage_funnel
  GROUP BY report_date, traffic_source
)
SELECT
  traffic_source,
  MAX(IF(report_date = DATE '{SNAP_END}', events_14d, 0)) AS latest_14d_events,
  MAX(IF(report_date = DATE '{SNAP_END}', purchases_14d, 0)) AS latest_14d_purchases,
  ROUND(AVG(SAFE_DIVIDE(cart_adds_14d * 100.0, NULLIF(product_views_14d, 0))), 2) AS view_to_cart_pct,
  ROUND(AVG(SAFE_DIVIDE(purchases_14d * 100.0, NULLIF(cart_adds_14d, 0))), 2) AS cart_to_purchase_pct,
  ROUND(AVG(SAFE_DIVIDE(purchases_14d * 100.0, NULLIF(product_views_14d, 0))), 2) AS product_to_purchase_pct
FROM daily_channel_totals
GROUP BY traffic_source
ORDER BY latest_14d_purchases DESC;
""",
    },
    {
        "id": 3,
        "engine": "SQL",
        "title": "6-Year Daily Inventory Capital-at-Risk & Rolling Turnover Ledger",
        "feature": "Heavy elastic range-join: physical SKUs × 2,373 calendar days (~600M active SKU-days)",
        "sql": rf"""
-- Query 3: 6-Year Daily Inventory Capital-at-Risk & Rolling Turnover Ledger
-- Purpose: Expand physical inventory lifecycles across 2,373 daily snapshots (~600M active SKU-days) to measure capital tied up and 30d sell-through.
WITH calendar_spine AS (
  SELECT snapshot_date
  FROM UNNEST(GENERATE_DATE_ARRAY('{SNAP_6_5Y_START}', '{SNAP_END}')) AS snapshot_date
),
daily_category_ledger AS (
  SELECT
    c.snapshot_date,
    ii.product_department AS department,
    ii.product_category AS category,
    COUNT(ii.id) AS units_on_shelf,
    SUM(ii.cost) AS capital_tied_up_usd,
    SUM(ii.product_retail_price - ii.cost) AS unrealized_margin_usd,
    AVG(DATE_DIFF(c.snapshot_date, DATE(ii.created_at), DAY)) AS avg_days_on_shelf,
    COUNTIF(DATE(ii.sold_at) = c.snapshot_date) AS units_sold_that_day
  FROM `{TABLES_FQN}.inventory_items` AS ii
  INNER JOIN calendar_spine AS c
    ON c.snapshot_date BETWEEN DATE(ii.created_at) AND COALESCE(DATE(ii.sold_at), DATE '{SNAP_END}')
  GROUP BY c.snapshot_date, department, category
),
rolling_metrics AS (
  SELECT
    department,
    category,
    snapshot_date,
    units_on_shelf,
    unrealized_margin_usd,
    avg_days_on_shelf,
    AVG(capital_tied_up_usd) OVER (
      PARTITION BY department, category
      ORDER BY snapshot_date
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30d_capital_usd,
    SUM(units_sold_that_day) OVER (
      PARTITION BY department, category
      ORDER BY snapshot_date
      ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
    ) AS rolling_30d_units_sold
  FROM daily_category_ledger
)
SELECT
  department,
  category,
  units_on_shelf AS current_unsold_units,
  ROUND(rolling_30d_capital_usd, 2) AS avg_30d_capital_tied_usd,
  ROUND(unrealized_margin_usd, 2) AS unrealized_margin_pool_usd,
  ROUND(avg_days_on_shelf, 1) AS avg_age_days,
  ROUND(SAFE_DIVIDE(rolling_30d_units_sold * 100.0, units_on_shelf), 2) AS monthly_sell_through_pct
FROM rolling_metrics
WHERE snapshot_date = DATE '{SNAP_END}'
ORDER BY avg_30d_capital_tied_usd DESC
LIMIT 10;
""",
    },
    {
        "id": 4,
        "engine": "SQL",
        "title": "5-Year Daily Rolling 30-Day Clickstream Abandonment & Velocity Cube",
        "feature": "High-concurrency sliding-window join: clickstream events × 1,826 daily 30d windows",
        "sql": rf"""
-- Query 4: 5-Year Daily Rolling 30-Day Clickstream Abandonment & Velocity Cube
-- Purpose: Evaluate 1,826 daily rolling 30-day windows over raw events to measure cart abandonment and volatility by channel & browser.
WITH daily_windows AS (
  SELECT report_date
  FROM UNNEST(GENERATE_DATE_ARRAY('{SNAP_5Y_START}', '{SNAP_END}')) AS report_date
),
daily_rolling_funnel AS (
  SELECT
    w.report_date,
    e.traffic_source,
    e.browser,
    COUNT(*) AS rolling_30d_events,
    COUNTIF(e.event_type = 'product') AS rolling_30d_product_views,
    COUNTIF(e.event_type = 'cart') AS rolling_30d_cart_events,
    COUNTIF(e.event_type = 'purchase') AS rolling_30d_purchases
  FROM `{TABLES_FQN}.events` AS e
  INNER JOIN daily_windows AS w
    ON DATE(e.created_at) BETWEEN DATE_SUB(w.report_date, INTERVAL 29 DAY) AND w.report_date
  GROUP BY w.report_date, e.traffic_source, e.browser
)
SELECT
  traffic_source,
  browser,
  ROUND(AVG(rolling_30d_events), 0) AS avg_30d_event_volume,
  MAX(IF(report_date = DATE '{SNAP_END}', rolling_30d_events, 0)) AS latest_30d_events,
  ROUND(AVG(SAFE_DIVIDE(rolling_30d_cart_events - rolling_30d_purchases, NULLIF(rolling_30d_cart_events, 0)) * 100.0), 2) AS avg_cart_abandonment_pct,
  ROUND(STDDEV(SAFE_DIVIDE(rolling_30d_purchases * 100.0, NULLIF(rolling_30d_product_views, 0))), 3) AS conversion_volatility_pts
FROM daily_rolling_funnel
GROUP BY traffic_source, browser
ORDER BY latest_30d_events DESC
LIMIT 10;
""",
    },
    {
        "id": 5,
        "engine": "SQL",
        "title": "6-Year Daily Rolling 90-Day Brand Profit & Return-Spike Stress Cube",
        "feature": "Distributed temporal join: order items × 2,191 daily rolling 90d brand windows",
        "sql": rf"""
-- Query 5: 6-Year Daily Rolling 90-Day Brand Profit & Return-Spike Stress Cube
-- Purpose: Evaluate 2,191 daily rolling 90-day windows per brand to benchmark current 90d profit against all-time peak and worst return-rate spikes.
WITH daily_snapshots AS (
  SELECT report_date
  FROM UNNEST(GENERATE_DATE_ARRAY('{SNAP_6Y_START}', '{SNAP_END}')) AS report_date
),
order_enriched AS (
  SELECT
    DATE(oi.created_at) AS order_date,
    oi.status,
    oi.sale_price,
    p.department,
    p.brand,
    p.cost
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
),
rolling_brand_health AS (
  SELECT
    s.report_date,
    oe.department,
    oe.brand,
    COUNT(*) AS rolling_90d_units,
    SUM(IF(oe.status NOT IN ('Cancelled', 'Returned'), oe.sale_price - oe.cost, 0)) AS rolling_90d_gross_profit,
    SAFE_DIVIDE(COUNTIF(oe.status = 'Returned') * 100.0, COUNT(*)) AS rolling_90d_return_pct
  FROM order_enriched AS oe
  INNER JOIN daily_snapshots AS s
    ON oe.order_date BETWEEN DATE_SUB(s.report_date, INTERVAL 89 DAY) AND s.report_date
  GROUP BY s.report_date, oe.department, oe.brand
  HAVING rolling_90d_units >= 20
)
SELECT
  department,
  brand,
  ROUND(MAX(IF(report_date = DATE '{SNAP_END}', rolling_90d_gross_profit, 0)), 2) AS current_90d_profit_usd,
  ROUND(MAX(rolling_90d_gross_profit), 2) AS all_time_peak_90d_profit_usd,
  ROUND(AVG(rolling_90d_return_pct), 2) AS historical_avg_return_pct,
  ROUND(MAX(rolling_90d_return_pct), 2) AS worst_90d_return_spike_pct
FROM rolling_brand_health
GROUP BY department, brand
HAVING MAX(rolling_90d_units) >= 60
ORDER BY current_90d_profit_usd DESC
LIMIT 10;
""",
    },
    {
        "id": 6,
        "engine": "GIS",
        "title": "Geodesic Last-Mile Shipping Distance vs. Local Micro-Hub Proximity",
        "feature": "16M+ spherical ST_DISTANCE evaluations across 1,200+ US micro-hub demand centroids",
        "sql": rf"""
-- Query 6: Geodesic Last-Mile Shipping Distance vs. Local Micro-Hub Proximity
-- Purpose: Benchmark actual warehouse shipping distances (avg, p90, % > 1,500 km) against 1,200+ local metro micro-hub centroids.
WITH metro_demand_nodes AS (
  SELECT
    CONCAT(city, ', ', state) AS metro_node,
    ST_CENTROID_AGG(ST_GEOGFROMTEXT(user_geom)) AS metro_geom
  FROM `{TABLES_FQN}.users`
  WHERE country = 'United States' AND city IS NOT NULL
  GROUP BY city, state
  HAVING COUNT(*) >= 5
),
shipment_distances AS (
  SELECT
    oi.id AS order_item_id,
    dc.name AS distribution_center,
    ST_DISTANCE(ST_GEOGFROMTEXT(dc.distribution_center_geom), ST_GEOGFROMTEXT(u.user_geom)) / 1000.0 AS actual_distance_km,
    DATETIME_DIFF(oi.delivered_at, oi.shipped_at, HOUR) / 24.0 AS transit_days,
    MIN(ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), m.metro_geom) / 1000.0) AS nearest_local_hub_km
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  INNER JOIN `{TABLES_FQN}.distribution_centers` AS dc
    ON p.distribution_center_id = dc.id
  INNER JOIN `{TABLES_FQN}.users` AS u
    ON oi.user_id = u.id
  CROSS JOIN metro_demand_nodes AS m
  WHERE u.country = 'United States'
    AND oi.delivered_at IS NOT NULL
    AND oi.shipped_at IS NOT NULL
  GROUP BY oi.id, dc.name, dc.distribution_center_geom, u.user_geom, oi.delivered_at, oi.shipped_at
)
SELECT
  distribution_center,
  COUNT(*) AS domestic_shipments,
  ROUND(AVG(actual_distance_km), 1) AS avg_distance_km,
  ROUND(APPROX_QUANTILES(actual_distance_km, 100)[OFFSET(90)], 1) AS p90_distance_km,
  ROUND(AVG(nearest_local_hub_km), 1) AS avg_local_hub_km,
  ROUND(AVG(transit_days), 2) AS avg_transit_days,
  ROUND(COUNTIF(actual_distance_km > 1500) * 100.0 / COUNT(*), 1) AS pct_long_haul_gt_1500km
FROM shipment_distances
GROUP BY distribution_center
ORDER BY avg_distance_km DESC;
""",
    },
    {
        "id": 7,
        "engine": "GIS",
        "title": "Optimal Nearest-Warehouse & Micro-Hub Re-Routing Simulation",
        "feature": "High-concurrency spatial nearest-neighbor across all US buyers × warehouses × city hubs",
        "sql": rf"""
-- Query 7: Optimal Nearest-Warehouse & Micro-Hub Re-Routing Simulation
-- Purpose: Simultaneously evaluate every US buyer against all 10 warehouses and regional micro-hubs to quantify reducible freight-km per category.
WITH city_micro_hubs AS (
  SELECT
    CONCAT(city, ', ', state) AS hub_label,
    ST_CENTROID_AGG(ST_GEOGFROMTEXT(user_geom)) AS hub_geom,
    COUNT(*) AS city_buyers
  FROM `{TABLES_FQN}.users`
  WHERE country = 'United States' AND city IS NOT NULL
  GROUP BY city, state
  HAVING city_buyers >= 15
),
nearest_hub_per_user AS (
  SELECT
    u.id AS user_id,
    MIN(ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), ST_GEOGFROMTEXT(dc.distribution_center_geom)) / 1000.0) AS nearest_dc_km,
    MIN(ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), mh.hub_geom) / 1000.0) AS nearest_microhub_km
  FROM `{TABLES_FQN}.users` AS u
  CROSS JOIN `{TABLES_FQN}.distribution_centers` AS dc
  CROSS JOIN city_micro_hubs AS mh
  WHERE u.country = 'United States'
  GROUP BY u.id
)
SELECT
  p.category,
  COUNT(oi.id) AS us_shipments,
  ROUND(AVG(ST_DISTANCE(ST_GEOGFROMTEXT(actual_dc.distribution_center_geom), ST_GEOGFROMTEXT(u.user_geom)) / 1000.0), 1) AS avg_actual_km,
  ROUND(AVG(nh.nearest_dc_km), 1) AS avg_nearest_dc_km,
  ROUND(AVG(nh.nearest_microhub_km), 1) AS avg_microhub_km,
  ROUND(SUM(
    GREATEST(0, (ST_DISTANCE(ST_GEOGFROMTEXT(actual_dc.distribution_center_geom), ST_GEOGFROMTEXT(u.user_geom)) / 1000.0) - nh.nearest_dc_km)
  ), 0) AS reducible_freight_km
FROM `{TABLES_FQN}.order_items` AS oi
INNER JOIN `{TABLES_FQN}.products` AS p
  ON oi.product_id = p.id
INNER JOIN `{TABLES_FQN}.distribution_centers` AS actual_dc
  ON p.distribution_center_id = actual_dc.id
INNER JOIN `{TABLES_FQN}.users` AS u
  ON oi.user_id = u.id
INNER JOIN nearest_hub_per_user AS nh
  ON u.id = nh.user_id
WHERE u.country = 'United States'
GROUP BY p.category
ORDER BY reducible_freight_km DESC
LIMIT 8;
""",
    },
    {
        "id": 8,
        "engine": "GIS",
        "title": "Multi-Year Rolling 300 km Catchment Polygon, Convex Hull & Centroid",
        "feature": "12,000+ daily geodesic ST_CONVEXHULL + ST_AREA + ST_CENTROID polygon aggregations",
        "sql": rf"""
-- Query 8: Multi-Year Rolling 300 km Catchment Polygon, Convex Hull & Centroid
-- Purpose: Compute daily rolling 180-day 300 km catchment revenue, geodesic convex-hull area (sq km), and demand centroid per warehouse.
WITH dated_customer_orders AS (
  SELECT
    DATE(oi.created_at) AS order_date,
    u.id AS user_id,
    ST_GEOGFROMTEXT(u.user_geom) AS user_pt,
    IF(oi.status NOT IN ('Cancelled', 'Returned'), oi.sale_price, 0) AS net_sale
  FROM `{TABLES_FQN}.users` AS u
  INNER JOIN `{TABLES_FQN}.order_items` AS oi
    ON u.id = oi.user_id
  WHERE u.country = 'United States'
),
daily_snapshots AS (
  SELECT report_date
  FROM UNNEST(GENERATE_DATE_ARRAY('{SNAP_3_5Y_START}', '{SNAP_END}')) AS report_date
),
daily_catchment_polygons AS (
  SELECT
    s.report_date,
    dc.name AS distribution_center,
    COUNT(DISTINCT dco.user_id) AS active_180d_buyers,
    SUM(dco.net_sale) AS rolling_180d_revenue,
    ST_AREA(ST_CONVEXHULL(ST_UNION_AGG(dco.user_pt))) / 1e6 AS hull_sq_km,
    ST_CENTROID(ST_UNION_AGG(dco.user_pt)) AS centroid_pt
  FROM `{TABLES_FQN}.distribution_centers` AS dc
  INNER JOIN dated_customer_orders AS dco
    ON ST_DWITHIN(ST_GEOGFROMTEXT(dc.distribution_center_geom), dco.user_pt, 300000)
  INNER JOIN daily_snapshots AS s
    ON dco.order_date BETWEEN DATE_SUB(s.report_date, INTERVAL 179 DAY) AND s.report_date
  GROUP BY s.report_date, dc.name
)
SELECT
  distribution_center,
  MAX(IF(report_date = DATE '{SNAP_END}', active_180d_buyers, 0)) AS current_180d_buyers_300km,
  ROUND(MAX(IF(report_date = DATE '{SNAP_END}', rolling_180d_revenue, 0)), 2) AS current_180d_catchment_rev,
  ROUND(AVG(hull_sq_km), 0) AS avg_catchment_hull_sq_km,
  ANY_VALUE(IF(report_date = DATE '{SNAP_END}', ST_ASTEXT(centroid_pt), NULL)) AS current_demand_centroid_wkt
FROM daily_catchment_polygons
GROUP BY distribution_center
ORDER BY current_180d_catchment_rev DESC;
""",
    },
    {
        "id": 9,
        "engine": "AI.*",
        "title": "Zero-Shot Multi-Series Weekly Revenue Forecast",
        "feature": "AI.FORECAST (TimesFM 2.5 foundation model)",
        "sql": rf"""
-- Query 9: Zero-Shot Multi-Series Weekly Revenue Forecast (AI.FORECAST + TimesFM 2.5)
-- Purpose: Forecast the next 4 weeks of revenue and 90% confidence intervals for Men's and Women's apparel with zero model training.
WITH weekly_dept_sales AS (
  SELECT
    DATE_TRUNC(DATE(oi.created_at), WEEK(MONDAY)) AS sales_week,
    p.department,
    SUM(oi.sale_price) AS weekly_revenue
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  WHERE oi.status NOT IN ('Cancelled', 'Returned')
    AND DATE(oi.created_at) BETWEEN '{FORECAST_START}' AND '{FORECAST_END}'
  GROUP BY sales_week, p.department
)
SELECT
  department,
  DATE(forecast_timestamp) AS forecast_week,
  ROUND(forecast_value, 2) AS predicted_revenue,
  ROUND(prediction_interval_lower_bound, 2) AS lower_bound_90,
  ROUND(prediction_interval_upper_bound, 2) AS upper_bound_90
FROM AI.FORECAST(
  TABLE weekly_dept_sales,
  data_col => 'weekly_revenue',
  timestamp_col => 'sales_week',
  id_cols => ['department'],
  horizon => 4,
  confidence_level => 0.90,
  model => 'TimesFM 2.5'
)
ORDER BY department, forecast_week;
""",
    },
    {
        "id": 10,
        "engine": "AI.*",
        "title": "Automated Daily Order Spike Detection",
        "feature": "AI.DETECT_ANOMALIES (TimesFM 2.5 foundation model)",
        "sql": rf"""
-- Query 10: Automated Daily Order Spike Detection (AI.DETECT_ANOMALIES + TimesFM 2.5)
-- Purpose: Detect statistically anomalous daily order volume surges in the latest 60 days against 16 months of historical baseline.
WITH daily_dept_orders AS (
  SELECT
    DATE(oi.created_at) AS order_date,
    p.department,
    COUNT(oi.id) AS daily_items
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  WHERE oi.status != 'Cancelled'
    AND DATE(oi.created_at) BETWEEN '{ANOM_HIST_START}' AND '{SNAP_END}'
  GROUP BY order_date, p.department
)
SELECT
  department,
  DATE(time_series_timestamp) AS order_date,
  CAST(time_series_data AS INT64) AS actual_items,
  ROUND(lower_bound, 1) AS expected_min,
  ROUND(upper_bound, 1) AS expected_max,
  ROUND(anomaly_probability, 4) AS anomaly_prob,
  is_anomaly
FROM AI.DETECT_ANOMALIES(
  (SELECT * FROM daily_dept_orders WHERE order_date < '{ANOM_CUTOFF}'),
  (SELECT * FROM daily_dept_orders WHERE order_date >= '{ANOM_CUTOFF}'),
  data_col => 'daily_items',
  timestamp_col => 'order_date',
  id_cols => ['department'],
  model => 'TimesFM 2.5',
  anomaly_prob_threshold => 0.80
)
ORDER BY anomaly_probability DESC
LIMIT 10;
""",
    },
    {
        "id": 11,
        "engine": "AI.*",
        "title": "6-Dimensional YoY Revenue Growth Root-Cause Analysis",
        "feature": "Clickstream engagement tiers + warehouse hubs → AI.KEY_DRIVERS",
        "sql": rf"""
-- Query 11: 6-Dimensional YoY Revenue Growth Root-Cause Analysis (AI.KEY_DRIVERS)
-- Purpose: Enrich orders with clickstream engagement tiers and fulfillment hubs to discover top YoY revenue growth drivers.
WITH user_engagement AS (
  SELECT
    user_id,
    CASE
      WHEN COUNT(*) >= 25 THEN 'Power_Browser_25+'
      WHEN COUNT(*) >= 10 THEN 'Active_Browser_10_24'
      ELSE 'Light_Browser_<10'
    END AS browse_tier
  FROM `{TABLES_FQN}.events`
  WHERE user_id IS NOT NULL
  GROUP BY user_id
),
yoy_sales AS (
  SELECT
    oi.sale_price,
    p.department,
    p.category,
    dc.name AS fulfillment_hub,
    u.country,
    u.traffic_source,
    COALESCE(ue.browse_tier, 'Guest_Checkout') AS browse_tier,
    (DATE(oi.created_at) BETWEEN '{YOY_CURR_START}' AND '{YOY_CURR_END}') AS is_current_6m
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  INNER JOIN `{TABLES_FQN}.distribution_centers` AS dc
    ON p.distribution_center_id = dc.id
  INNER JOIN `{TABLES_FQN}.users` AS u
    ON oi.user_id = u.id
  LEFT JOIN user_engagement AS ue
    ON u.id = ue.user_id
  WHERE oi.status NOT IN ('Cancelled', 'Returned')
    AND (
      DATE(oi.created_at) BETWEEN '{YOY_PREV_START}' AND '{YOY_PREV_END}'
      OR DATE(oi.created_at) BETWEEN '{YOY_CURR_START}' AND '{YOY_CURR_END}'
    )
)
SELECT
  ARRAY_TO_STRING(drivers, ' | ') AS segment_driver,
  ROUND(metric_reference, 2) AS prior_6m_revenue,
  ROUND(metric_interest, 2) AS current_6m_revenue,
  ROUND(difference, 2) AS yoy_delta,
  ROUND(relative_difference * 100.0, 1) AS yoy_growth_pct,
  ROUND(apriori_support * 100.0, 1) AS population_share_pct
FROM AI.KEY_DRIVERS(
  TABLE yoy_sales,
  metric_col => 'sale_price',
  dimension_cols => ['department', 'category', 'fulfillment_hub', 'country', 'traffic_source', 'browse_tier'],
  interest_label_col => 'is_current_6m',
  top_k => 8
)
ORDER BY yoy_delta DESC;
""",
    },
    {
        "id": 12,
        "engine": "AI.*",
        "title": "Zero-Shot Semantic Filtering & Campaign Persona Tagging",
        "feature": "AI.IF (semantic WHERE filter) + AI.CLASSIFY",
        "sql": rf"""
-- Query 12: Zero-Shot Semantic Filtering & Campaign Persona Tagging (AI.IF + AI.CLASSIFY)
-- Purpose: Semantically filter top outerwear SKUs for true cold/wet-weather gear and classify each into a marketing persona.
WITH top_outerwear AS (
  SELECT
    p.id AS product_id,
    p.brand,
    p.category,
    p.name AS product_name,
    ROUND(AVG(p.retail_price), 2) AS retail_price,
    ROUND(SUM(oi.sale_price), 2) AS realized_revenue
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  WHERE oi.status NOT IN ('Cancelled', 'Returned')
    AND p.category IN ('Outerwear & Coats', 'Active')
  GROUP BY p.id, p.brand, p.category, p.name
  ORDER BY realized_revenue DESC
  LIMIT 15
)
SELECT
  product_id,
  brand,
  product_name,
  retail_price,
  realized_revenue,
  AI.CLASSIFY(
    product_name,
    categories => [
      'Alpine & Heavy Winter',
      'Urban Tailored Coat',
      'Rain & Wind Shell',
      'Fleece & Mid-Layer'
    ]
  ) AS campaign_segment
FROM top_outerwear
WHERE AI.IF((product_name, 'Is this garment intended for cold weather, winter warmth, or rain/wind protection?'))
ORDER BY realized_revenue DESC;
""",
    },
    {
        "id": 13,
        "engine": "AI.*",
        "title": "Unified SQL + Micro-Hub GIS + Structured GenAI Supply-Chain Brief",
        "feature": "Geodesic micro-hub gap simulation + AI.GENERATE (output_schema)",
        "sql": rf"""
-- Query 13: Unified SQL + Micro-Hub GIS + Structured GenAI Supply-Chain Brief (AI.GENERATE)
-- Purpose: Combine warehouse shipping KPIs with regional micro-hub distance benchmarks and synthesize typed executive action plans.
WITH metro_nodes AS (
  SELECT
    ST_CENTROID_AGG(ST_GEOGFROMTEXT(user_geom)) AS metro_geom
  FROM `{TABLES_FQN}.users`
  WHERE country = 'United States' AND city IS NOT NULL
  GROUP BY city, state
  HAVING COUNT(*) >= 10
),
user_hub_gap AS (
  SELECT
    u.id AS user_id,
    u.user_geom,
    MIN(ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), m.metro_geom) / 1000.0) AS nearest_metro_km
  FROM `{TABLES_FQN}.users` AS u
  CROSS JOIN metro_nodes AS m
  WHERE u.country = 'United States'
  GROUP BY u.id, u.user_geom
),
dc_telemetry AS (
  SELECT
    dc.name AS distribution_center,
    ROUND(SUM(IF(oi.status NOT IN ('Cancelled', 'Returned'), oi.sale_price, 0)), 0) AS net_revenue_usd,
    ROUND(COUNTIF(oi.status = 'Returned') * 100.0 / COUNT(oi.id), 2) AS return_rate_pct,
    ROUND(AVG(ST_DISTANCE(ST_GEOGFROMTEXT(dc.distribution_center_geom), ST_GEOGFROMTEXT(ug.user_geom)) / 1000.0), 0) AS avg_ship_km,
    ROUND(AVG(ug.nearest_metro_km), 0) AS potential_microhub_km,
    ROUND(AVG(
      IF(oi.delivered_at IS NOT NULL AND oi.shipped_at IS NOT NULL,
         DATETIME_DIFF(oi.delivered_at, oi.shipped_at, HOUR) / 24.0,
         NULL)
    ), 2) AS avg_transit_days
  FROM `{TABLES_FQN}.order_items` AS oi
  INNER JOIN `{TABLES_FQN}.products` AS p
    ON oi.product_id = p.id
  INNER JOIN `{TABLES_FQN}.distribution_centers` AS dc
    ON p.distribution_center_id = dc.id
  INNER JOIN user_hub_gap AS ug
    ON oi.user_id = ug.user_id
  GROUP BY dc.name
)
SELECT
  distribution_center,
  net_revenue_usd,
  avg_ship_km,
  potential_microhub_km,
  return_rate_pct,
  ai_plan.priority_tier,
  ai_plan.bottleneck_diagnosis,
  ai_plan.recommended_action
FROM (
  SELECT
    *,
    AI.GENERATE(
      CONCAT(
        'Analyze this US e-commerce warehouse KPI summary and recommend one concrete supply-chain action. ',
        'Warehouse: ', distribution_center,
        ', Net Revenue: $', CAST(net_revenue_usd AS STRING),
        ', Avg Shipping Distance: ', CAST(avg_ship_km AS STRING), ' km (vs ', CAST(potential_microhub_km AS STRING), ' km via regional micro-hubs)',
        ', Avg Transit Days: ', CAST(avg_transit_days AS STRING),
        ', Return Rate: ', CAST(return_rate_pct AS STRING), '%.'
      ),
      output_schema => 'priority_tier STRING, bottleneck_diagnosis STRING, recommended_action STRING'
    ) AS ai_plan
  FROM dc_telemetry
)
ORDER BY avg_ship_km DESC;
""",
    },
    {
        "id": 14,
        "engine": "Graph",
        "title": "8-Hop Repeat-Buyer Category & Warehouse Transition Graph (ISO GQL)",
        "feature": "Standalone GRAPH ... MATCH p = ... RETURN + PATH_LENGTH(p) across 5 entity types",
        "sql": rf"""
-- Query 14: 8-Hop Repeat-Buyer Category & Warehouse Transition Graph (Native ISO GQL)
-- Purpose: Traverse an 8-hop path across sequential customer orders (Warehouse -> Product -> OrderItem -> Order -> User <- Order <- OrderItem <- Product <- Warehouse) to uncover repeat-purchase category bridges and cross-hub fulfillment switches.
GRAPH `{GRAPH_DATASET_FQN}.ecommerce_graph`
MATCH p = (dc1:DistributionCenter)<-[:product_stocked_at]-(p1:Product)<-[:includes_product]-(oi1:OrderItem)
          -[:belongs_to_order]->(o1:`Order`)-[:placed_order]->(u:User)<-[:placed_order]-(o2:`Order`)
          <-[:belongs_to_order]-(oi2:OrderItem)-[:includes_product]->(p2:Product)-[:product_stocked_at]->(dc2:DistributionCenter)
WHERE o1.created_at < o2.created_at
  AND p1.category != p2.category
  AND o1.status != 'Cancelled'
  AND o2.status != 'Cancelled'
RETURN
  p1.category AS first_order_category,
  p2.category AS repeat_order_category,
  MAX(PATH_LENGTH(p)) AS graph_hops,
  COUNT(DISTINCT u.id) AS repeat_customers,
  COUNT(*) AS cross_order_transitions,
  COUNTIF(dc1.id != dc2.id) AS cross_hub_switches,
  ROUND(AVG(DATETIME_DIFF(o2.created_at, o1.created_at, DAY)), 1) AS avg_days_to_next_order,
  ROUND(SUM(oi2.sale_price), 2) AS downstream_repeat_revenue
ORDER BY cross_order_transitions DESC, downstream_repeat_revenue DESC
LIMIT 12;
""",
    },
    {
        "id": 15,
        "engine": "All-in-One",
        "title": "4-Engine Convergence: SQL + Property Graph + Spatial GIS + Gemini AI.*",
        "feature": "GRAPH_TABLE + clickstream & warehouse stock SQL + spherical GIS micro-hubs & hulls + AI.GENERATE",
        "sql": rf"""
-- Query 15: 4-Engine Convergence — Property Graph + Relational SQL + Spatial GIS + Gemini AI.GENERATE
-- Purpose: Combine a 4-hop Property Graph traversal with relational clickstream & inventory SQL, spherical GIS micro-hub & convex-hull geometry, and Gemini AI.GENERATE across all 7 tables in one query.
WITH graph_fulfillment_paths AS (
  -- 1. GRAPH: Traverse 4-hop fulfillment paths across all 5 node types in ecommerce_graph
  SELECT
    user_id,
    user_state,
    user_geom,
    order_id,
    item_status,
    sale_price,
    unit_cost,
    category,
    hub_id,
    hub_name,
    hub_geom
  FROM GRAPH_TABLE(
    `{GRAPH_DATASET_FQN}.ecommerce_graph`
    MATCH (u:User WHERE u.country = 'United States')<-[:placed_order]-(o:`Order` WHERE o.status != 'Cancelled')
          <-[:belongs_to_order]-(oi:OrderItem)-[:includes_product]->(p:Product)
          -[:product_stocked_at]->(dc:DistributionCenter)
    COLUMNS (
      u.id AS user_id,
      u.state AS user_state,
      u.user_geom AS user_geom,
      o.order_id AS order_id,
      oi.status AS item_status,
      oi.sale_price AS sale_price,
      p.cost AS unit_cost,
      p.category AS category,
      dc.id AS hub_id,
      dc.name AS hub_name,
      dc.distribution_center_geom AS hub_geom
    )
  )
),
user_clickstream AS (
  -- 2a. SQL (Relational Clickstream): Aggregate web events (outside the graph) per customer
  SELECT
    user_id,
    COUNT(*) AS web_events,
    COUNTIF(event_type = 'cart') AS cart_adds,
    COUNTIF(event_type = 'purchase') AS web_purchases
  FROM `{TABLES_FQN}.events`
  WHERE user_id IS NOT NULL
  GROUP BY user_id
),
hub_unsold_stock AS (
  -- 2b. SQL (Relational Inventory): Unsold warehouse capital tied up per hub & category
  SELECT
    product_distribution_center_id AS hub_id,
    product_category AS category,
    ROUND(SUM(cost), 2) AS unsold_stock_usd
  FROM `{TABLES_FQN}.inventory_items`
  WHERE sold_at IS NULL
  GROUP BY hub_id, category
),
us_metro_nodes AS (
  -- 3a. GIS: Derive regional US micro-hub centroids from buyer geography
  SELECT
    CONCAT(city, ', ', state) AS metro_hub,
    ST_CENTROID_AGG(ST_GEOGFROMTEXT(user_geom)) AS metro_pt
  FROM `{TABLES_FQN}.users`
  WHERE country = 'United States' AND city IS NOT NULL AND user_geom IS NOT NULL
  GROUP BY city, state
  HAVING COUNT(*) >= 12
),
buyer_nearest_microhub AS (
  -- 3b. GIS Fan-Out: Match every US buyer to their closest regional micro-hub via spherical ST_DISTANCE
  SELECT
    u.id AS user_id,
    ARRAY_AGG(
      STRUCT(
        m.metro_hub,
        ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), m.metro_pt) / 1000.0 AS microhub_km
      )
      ORDER BY ST_DISTANCE(ST_GEOGFROMTEXT(u.user_geom), m.metro_pt) ASC
      LIMIT 1
    )[OFFSET(0)] AS best_local
  FROM `{TABLES_FQN}.users` AS u
  CROSS JOIN us_metro_nodes AS m
  WHERE u.country = 'United States' AND u.user_geom IS NOT NULL
  GROUP BY u.id
),
corridor_convergence AS (
  -- Converge GRAPH paths + SQL clickstream/inventory + GIS haul gap & convex-hull area + SQL window rank
  SELECT
    g.hub_name AS origin_hub,
    g.user_state AS dest_state,
    g.category,
    COUNT(*) AS graph_shipped_items,
    ROUND(SUM(g.sale_price - g.unit_cost), 2) AS gross_margin_usd,
    ROUND(100.0 * COUNTIF(g.item_status = 'Returned') / COUNT(*), 2) AS return_rate_pct,
    ROUND(AVG(cs.web_events), 1) AS avg_buyer_clicks,
    COALESCE(MAX(inv.unsold_stock_usd), 0.0) AS hub_unsold_capital_usd,
    ROUND(AVG(ST_DISTANCE(ST_GEOGFROMTEXT(g.user_geom), ST_GEOGFROMTEXT(g.hub_geom)) / 1000.0), 0) AS avg_haul_km,
    ROUND(AVG(bm.best_local.microhub_km), 0) AS avg_microhub_km,
    APPROX_TOP_COUNT(bm.best_local.metro_hub, 1)[OFFSET(0)].value AS top_forward_microhub,
    ROUND(ST_AREA(ST_CONVEXHULL(ST_UNION_AGG(ST_GEOGFROMTEXT(g.user_geom)))) / 1e6, 0) AS demand_hull_sq_km,
    RANK() OVER (
      ORDER BY SUM(g.sale_price - g.unit_cost) * (1.0 + COUNTIF(g.item_status = 'Returned') / COUNT(*)) DESC
    ) AS corridor_rank
  FROM graph_fulfillment_paths AS g
  LEFT JOIN user_clickstream AS cs
    ON g.user_id = cs.user_id
  LEFT JOIN hub_unsold_stock AS inv
    ON g.hub_id = inv.hub_id AND g.category = inv.category
  INNER JOIN buyer_nearest_microhub AS bm
    ON g.user_id = bm.user_id
  WHERE g.user_geom IS NOT NULL AND g.hub_geom IS NOT NULL
  GROUP BY g.hub_name, g.user_state, g.category
  HAVING graph_shipped_items >= 25 AND avg_haul_km >= 1800
  QUALIFY corridor_rank <= 6
)
-- 4. AI.*: Synthesize Graph + SQL + GIS signals into typed executive actions via Gemini
SELECT
  origin_hub,
  dest_state,
  category,
  graph_shipped_items,
  gross_margin_usd,
  return_rate_pct,
  avg_buyer_clicks,
  hub_unsold_capital_usd,
  avg_haul_km,
  avg_microhub_km,
  top_forward_microhub,
  demand_hull_sq_km,
  ai_plan.priority_tier,
  ai_plan.corridor_diagnosis,
  ai_plan.forward_hub_plan,
  ai_plan.cx_retention_play
FROM (
  SELECT
    *,
    AI.GENERATE(
      CONCAT(
        'Synthesize a concise supply-chain & CX action plan (under 22 words per field; priority_tier must be P0, P1, or P2). ',
        'Graph Corridor: ', origin_hub, ' -> ', dest_state, ' (', category, ', ', CAST(graph_shipped_items AS STRING), ' items, $', CAST(gross_margin_usd AS STRING), ' margin, ', CAST(return_rate_pct AS STRING), '% returns). ',
        'SQL Signals: ', CAST(avg_buyer_clicks AS STRING), ' avg web clicks/buyer, $', CAST(hub_unsold_capital_usd AS STRING), ' unsold category stock at origin hub. ',
        'GIS Signals: ', CAST(avg_haul_km AS STRING), ' km actual haul vs ', CAST(avg_microhub_km AS STRING), ' km via forward hub ', top_forward_microhub, ' across ', CAST(demand_hull_sq_km AS STRING), ' sq km hull.'
      ),
      output_schema => 'priority_tier STRING, corridor_diagnosis STRING, forward_hub_plan STRING, cx_retention_play STRING'
    ) AS ai_plan
  FROM corridor_convergence
)
ORDER BY corridor_rank ASC;
""",
    },
]

print(
    f"📋 Loaded {len(QUERIES)} queries targeting `{TABLES_FQN}` + `{GRAPH_DATASET_FQN}.ecommerce_graph` (5 SQL, 3 GIS, 5 AI.*, 1 Graph, 1 All-in-One)."
)

In [ ]:
# @title 2. Run All 15 Queries in Parallel
RESULTS, BATCH_ID = run_parallel_suite(QUERIES)

In [ ]:
# @title 3. Visual Insights Grid
render_visual_insights_grid(RESULTS)

In [ ]:
# @title 4. Raw Results & SQL
render_results_accordion(RESULTS)

---
## Conversational Graph Agent
Open `ecommerce_graph` in BigQuery Studio, click **Chat** → **Create Agent**, and ask multi-hop customer, product, and fulfillment questions in plain English.

In [ ]:
# @title 5. Chat with the Property Graph
_GRAPH_STUDIO_URL = (
    f"https://console.cloud.google.com/bigquery?project={urllib.parse.quote(PROJECT_ID)}"
    f"&ws=!1m5!1m4!18m3!1s{urllib.parse.quote(PROJECT_ID)}!2s{urllib.parse.quote(GRAPH_DATASET_ID)}!3secommerce_graph"
)

_GRAPH_PROMPTS = [
    (
        "Semantic Measures & Fulfillment Rollup",
        "Which top 5 distribution centers generate the highest total revenue and order count across non-cancelled, non-returned items, and what are their top product categories?",
    ),
    (
        "Multi-Hop Acquisition Channel Economics",
        "Compare total revenue, total cost, and user count by customer traffic_source across the Men and Women departments.",
    ),
    (
        "Cross-Hub Regional Corridor Demand",
        "Which US customer states order the highest item_count of products stocked in the Chicago IL and Los Angeles CA distribution centers?",
    ),
    (
        "5-Entity Relationship Exploration",
        "Trace how top-spending users in California connect through orders and order items to products and the distribution centers that stock them.",
    ),
]

_prompt_cards_html = "".join(
    f'''
    <div style="background:#f8fafc;border:1px solid #cbd5e1;border-left:4px solid #0d9488;border-radius:8px;padding:10px 12px;">
      <div style="font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.4px;color:#0f766e;margin-bottom:4px;">
        Prompt {idx} · {html.escape(tag)}
      </div>
      <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12.5px;color:#0f172a;line-height:1.45;user-select:all;">
        {html.escape(q_text)}
      </div>
    </div>
    '''
    for idx, (tag, q_text) in enumerate(_GRAPH_PROMPTS, start=1)
)

display(HTML(f'''
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
            background:#ffffff;color:#0f172a;border:1.5px solid #99f6e4;border-top:5px solid #0d9488;
            border-radius:12px;padding:18px 20px;margin:8px 0;box-shadow:0 2px 6px rgba(13,148,136,0.08);">
  <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:12px;margin-bottom:12px;">
    <div>
      <div style="font-size:11.5px;font-weight:800;text-transform:uppercase;letter-spacing:0.6px;color:#0f766e;">
        💬 Conversational Graph Agent — Native Conversational Analytics on Property Graph
      </div>
      <div style="font-size:17px;font-weight:800;color:#0f172a;margin-top:3px;">
        Talk to Data &amp; Talk to Graph on <code>{html.escape(GRAPH_DATASET_FQN)}.ecommerce_graph</code>
      </div>
    </div>
    <a href="{html.escape(_GRAPH_STUDIO_URL)}" target="_blank" rel="noopener noreferrer"
       style="display:inline-flex;align-items:center;gap:6px;background:#0d9488;color:#ffffff;
              text-decoration:none;font-weight:700;font-size:13px;padding:9px 15px;border-radius:8px;
              box-shadow:0 1px 2px rgba(0,0,0,0.12);white-space:nowrap;">
      🕸️ Open <code>ecommerce_graph</code> in BigQuery Studio → Click "Chat" → "Create Agent" ↗
    </a>
  </div>

  <div style="font-size:13px;color:#334155;line-height:1.55;margin-bottom:14px;">
    Open <code>ecommerce_graph</code> in BigQuery Studio and click <b>Chat</b> → <b>Create Agent</b> in the toolbar.
    Because the property graph declares explicit directed edges
    (<code>OrderItem → Order → User</code> and <code>OrderItem → Product → DistributionCenter</code>), rich column
    <code>OPTIONS(description=...)</code>, and <b>6 grain-locked semantic <code>MEASURE(...)</code> aggregations</b>
    (<code>total_revenue</code>, <code>item_count</code>, <code>order_count</code>, <code>user_count</code>,
    <code>total_cost</code>, <code>product_count</code>), the Graph Agent answers multi-hop business questions without
    join hallucinations or fanout double-counting:
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:10px;">
    {_prompt_cards_html}
  </div>
</div>
'''))

---
## Storage & Telemetry
How much data is physically stored inside BigQuery to power the 15 parallel `SQL`, `GIS`, `AI.*`, `Graph`, and `All-in-One` queries — **plus the live Conversational Graph Agent** — and where do the tables in `thelook_catalog` actually live?

In [ ]:
# @title 6. Dataset Storage & Table Origins
REVEAL_STORAGE_SQL = rf"""
-- Check physical bytes stored in BigQuery dataset {GRAPH_DATASET_FQN} (0 tables, ecommerce_graph only)
SELECT
  t.table_id AS object_name,
  CASE t.type WHEN 1 THEN 'BASE TABLE' ELSE 'OTHER' END AS object_type,
  t.size_bytes AS bytes_stored_in_bigquery,
  'BigQuery table' AS definition_summary
FROM `{GRAPH_DATASET_FQN}.__TABLES__` AS t
UNION ALL
SELECT
  g.property_graph_name AS object_name,
  'PROPERTY GRAPH' AS object_type,
  0 AS bytes_stored_in_bigquery,
  CONCAT(
    REGEXP_EXTRACT(g.ddl, r'(CREATE PROPERTY GRAPH [^\n]+)'),
    ' -> ',
    REGEXP_EXTRACT(g.ddl, r'NODE TABLES \(\n([^\n]+)')
  ) AS definition_summary
FROM `{GRAPH_DATASET_FQN}.INFORMATION_SCHEMA.PROPERTY_GRAPHS` AS g
ORDER BY object_type DESC, object_name;
"""

df_storage = _job_to_df(client.query(REVEAL_STORAGE_SQL, location=LOCATION))
total_bq_bytes = int(df_storage["bytes_stored_in_bigquery"].sum())
table_cnt = int((df_storage["object_type"] != "PROPERTY GRAPH").sum())

# Resolve live physical storage URIs and external catalog origin for `{TABLES_FQN}.*`
_origin_rows: list[dict] = []
try:
    _rev_sess = _kc_api_session()
    _rev_base = "https://dataplex.googleapis.com/v1"
    _rev_q = urllib.parse.urlencode({
        "query": f"(system=BIGLAKE AND name:{CATALOG_ID})",
        "scope": f"projects/{PROJECT_ID}",
        "pageSize": 50,
    })
    _rev_sr = _rev_sess.post(
        f"{_rev_base}/projects/{PROJECT_ID}/locations/global:searchEntries?{_rev_q}",
        data=b"",
        timeout=30,
    )
    _rev_items = (_rev_sr.json().get("results") or []) if _rev_sr.ok else []
    _tbl_entries = sorted({
        ((it.get("dataplexEntry") or {}).get("name") or "")
        for it in _rev_items
        if "/@biglake/" in ((it.get("dataplexEntry") or {}).get("name") or "")
        and "/tables/" in ((it.get("dataplexEntry") or {}).get("name") or "")
        and f"/locations/{LOCATION}/" in ((it.get("dataplexEntry") or {}).get("name") or "")
    })

    def _lookup_origin(ename: str) -> dict | None:
        lq = urllib.parse.urlencode({"entry": ename, "view": "ALL"})
        lr = _rev_sess.get(
            f"{_rev_base}/projects/{PROJECT_ID}/locations/{LOCATION}:lookupEntry?{lq}",
            timeout=30,
        )
        if not lr.ok:
            return None
        payload = lr.json() or {}
        aspects = payload.get("aspects") or {}
        bl = _find_aspect_data(aspects, "global.biglake-table")
        st = _find_aspect_data(aspects, "global.storage")
        tname = (payload.get("entrySource") or {}).get("displayName") or ename.rsplit("/", 1)[-1]
        uris = st.get("sourceUris") or []
        return {
            "table_fqn": f"{TABLES_FQN}.{tname}",
            "bytes_in_bigquery": 0,
            "external_catalog_origin": str(bl.get("catalogSource") or "EXTERNAL_CATALOG"),
            "open_table_format": str(bl.get("storageFormat") or st.get("format") or "OPEN_TABLE"),
            "physical_storage_uri": str(uris[0] if uris else (payload.get("entrySource") or {}).get("resource") or ""),
        }

    with ThreadPoolExecutor(max_workers=8) as _rev_pool:
        _origin_rows = [r for r in _rev_pool.map(_lookup_origin, _tbl_entries) if r]
except Exception:
    _origin_rows = []

df_origins = pd.DataFrame(_origin_rows)
_origins_block_html = ""
if not df_origins.empty:
    _cat_origins = ", ".join(sorted(set(df_origins["external_catalog_origin"].astype(str))))
    _tbl_formats = ", ".join(sorted(set(df_origins["open_table_format"].astype(str))))
    _origins_block_html = f'''
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
            background:#fff7ed;color:#9a3412;border:2px solid #fb923c;border-radius:10px;
            padding:14px 18px;margin:14px 0 10px 0;font-size:14.5px;">
  🌍 <b>Live Federated Table Origin (<code>{html.escape(TABLES_FQN)}.*</code>):</b>
  <b>{len(df_origins)} tables</b> federated directly from <code>{html.escape(_cat_origins)}</code>
  (format: <code>{html.escape(_tbl_formats)}</code>):
</div>
''' + df_origins.to_html(index=False, border=0, classes="bq-reveal-tbl")

display(HTML(f'''
<style>
  table.bq-reveal-tbl {{ width:100%; border-collapse:collapse; font-size:13px; background:#ffffff; color:#0f172a; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif; }}
  table.bq-reveal-tbl th {{ background:#f1f5f9; color:#334155; padding:8px 12px; text-align:left; border-bottom:2px solid #cbd5e1; }}
  table.bq-reveal-tbl td {{ padding:8px 12px; border-bottom:1px solid #e2e8f0; color:#0f172a; font-family:ui-monospace,SFMono-Regular,Menlo,monospace; font-size:12.5px; }}
</style>
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
            background:#fef7e0;color:#7c4a03;border:2px solid #fbbc04;border-radius:10px;
            padding:14px 18px;margin-bottom:10px;font-size:15px;">
  🔎 <b>Physical Bytes Stored in BigQuery (<code>{html.escape(GRAPH_DATASET_FQN)}</code>):
  <code>{total_bq_bytes:,} bytes</code></b> ({table_cnt} tables stored in BigQuery — only zero-copy <code>ecommerce_graph</code>):
</div>
''' + df_storage.to_html(index=False, border=0, classes="bq-reveal-tbl") + _origins_block_html))

In [ ]:
# @title 7. Cache Analysis
_REGION_SCHEMA = f"{PROJECT_ID}.region-{LOCATION.lower()}.INFORMATION_SCHEMA.JOBS_BY_PROJECT"
_LOGICAL_CATALOG_SQL = " UNION ALL\n  ".join(
    f"SELECT '{tbl}' AS table_id, {int(sz)} AS full_logical_bytes"
    for tbl, sz in TABLE_LOGICAL_BYTES.items()
)

target_batch_sql = (
    f"'{BATCH_ID}'"
    if "BATCH_ID" in globals() and BATCH_ID
    else f"(SELECT MAX((SELECT l.value FROM UNNEST(j2.labels) AS l WHERE l.key = 'demo_batch')) "
         f"FROM `{_REGION_SCHEMA}` AS j2 "
         f"WHERE j2.creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 HOUR))"
)

REVEAL_JOBS_SQL = rf"""
-- Audit slot elasticity and the 4-stage data reduction pipeline:
-- Logical Data -> Compressed Data -> Cross-Cloud Cache -> Actually Sent Over the Wire
WITH target_batch AS (
  SELECT {target_batch_sql} AS batch_id
),
table_logical_catalog AS (
  {_LOGICAL_CATALOG_SQL}
),
job_metrics AS (
  SELECT
    UPPER(
      COALESCE(
        (SELECT l.value FROM UNNEST(labels) AS l WHERE l.key = 'query_num'),
        'query'
      )
    ) AS query_num,
    REGEXP_EXTRACT(query, r'-- Query \d+: ([^\n]+)') AS query_title,
    creation_time,
    ROUND(TIMESTAMP_DIFF(end_time, start_time, MILLISECOND) / 1000.0, 2) AS elapsed_sec,
    ROUND(total_slot_ms / 1000.0, 1) AS slot_sec,
    ROUND(total_slot_ms / NULLIF(TIMESTAMP_DIFF(end_time, start_time, MILLISECOND), 0), 1) AS avg_slots,
    COALESCE((
      SELECT SUM(tlc.full_logical_bytes)
      FROM (SELECT DISTINCT rt.table_id FROM UNNEST(referenced_tables) AS rt) AS d
      LEFT JOIN table_logical_catalog AS tlc USING (table_id)
    ), 0) AS table_logical_bytes,
    COALESCE(total_bytes_processed, 0) AS raw_logical_scan_bytes,
    COALESCE((SELECT SUM(COALESCE(s.cache_bytes_read, 0)) FROM UNNEST(object_storage_stats) AS s), 0) AS cache_bytes,
    COALESCE((SELECT SUM(COALESCE(s.object_storage_bytes_read, 0)) FROM UNNEST(object_storage_stats) AS s), 0) AS wire_bytes,
    COALESCE((SELECT STRING_AGG(DISTINCT s.cloud_provider, ', ') FROM UNNEST(object_storage_stats) AS s), 'LOCAL') AS remote_cloud,
    ARRAY_TO_STRING(
      ARRAY(
        SELECT DISTINCT CONCAT(rt.dataset_id, '.', rt.table_id)
        FROM UNNEST(referenced_tables) AS rt
        ORDER BY 1
      ),
      ', '
    ) AS actual_physical_tables_read
  FROM `{_REGION_SCHEMA}`
  CROSS JOIN target_batch AS tb
  WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 HOUR)
    AND statement_type = 'SELECT'
    AND IF(
      tb.batch_id IS NOT NULL,
      EXISTS (SELECT 1 FROM UNNEST(labels) AS l WHERE l.key = 'demo_batch' AND l.value = tb.batch_id),
      (query LIKE '%{TABLES_FQN}.%' OR query LIKE '%{GRAPH_DATASET_FQN}.%')
        AND query NOT LIKE '%INFORMATION_SCHEMA%'
        AND query NOT LIKE '%__TABLES__%'
    )
  QUALIFY ROW_NUMBER() OVER (PARTITION BY query_num, query_title ORDER BY creation_time DESC) = 1
),
enriched AS (
  SELECT
    *,
    (cache_bytes + wire_bytes) AS compressed_bytes,
    COALESCE(
      NULLIF(raw_logical_scan_bytes, 0),
      CAST(
        ROUND(
          (cache_bytes + wire_bytes) * COALESCE(
            SAFE_DIVIDE(
              SUM(IF(raw_logical_scan_bytes > 0, raw_logical_scan_bytes, 0)) OVER (),
              NULLIF(SUM(IF(raw_logical_scan_bytes > 0, cache_bytes + wire_bytes, 0)) OVER (), 0)
            ),
            1.0
          )
        ) AS INT64
      )
    ) AS logical_scan_bytes
  FROM job_metrics
)
SELECT
  query_num,
  query_title,
  elapsed_sec,
  slot_sec,
  avg_slots,
  ROUND(table_logical_bytes / 1048576.0, 2) AS table_logical_mb,
  ROUND(logical_scan_bytes / 1048576.0, 2) AS logical_scan_mb,
  ROUND(compressed_bytes / 1048576.0, 2) AS compressed_mb,
  ROUND(SAFE_DIVIDE(logical_scan_bytes, NULLIF(compressed_bytes, 0)), 2) AS compression_ratio_num,
  FORMAT('%.2fx', COALESCE(SAFE_DIVIDE(logical_scan_bytes, NULLIF(compressed_bytes, 0)), 1.0)) AS compression_ratio,
  ROUND(cache_bytes / 1048576.0, 2) AS cache_hit_mb,
  ROUND(100.0 * SAFE_DIVIDE(cache_bytes, NULLIF(compressed_bytes, 0)), 1) AS cache_hit_pct,
  ROUND(wire_bytes / 1048576.0, 2) AS sent_over_wire_mb,
  remote_cloud,
  actual_physical_tables_read
FROM enriched
ORDER BY CAST(REGEXP_EXTRACT(query_num, r'\d+') AS INT64), creation_time DESC
LIMIT 20;
"""

df_jobs = _job_to_df(client.query(REVEAL_JOBS_SQL, location=LOCATION))

if df_jobs.empty:
    display(HTML('<div style="padding:12px;color:#b06000;background:#fef7e0;border-radius:8px;">No recent batch jobs found in JOBS_BY_PROJECT yet — run Step 2 first (and wait ~2s for telemetry ingestion).</div>'))
else:
    tot_slot_sec = float(df_jobs["slot_sec"].fillna(0).sum())
    tot_table_logical = float(df_jobs["table_logical_mb"].fillna(0).sum())
    tot_logical_scan = float(df_jobs["logical_scan_mb"].fillna(0).sum())
    tot_compressed = float(df_jobs["compressed_mb"].fillna(0).sum())
    tot_cache = float(df_jobs["cache_hit_mb"].fillna(0).sum())
    tot_wire = float(df_jobs["sent_over_wire_mb"].fillna(0).sum())

    col_prune_pct = 100.0 * (1.0 - (tot_logical_scan / max(tot_table_logical, 1e-6)))
    fleet_comp_ratio = tot_logical_scan / max(tot_compressed, 1e-6)
    peak_comp_ratio = float(df_jobs["compression_ratio_num"].fillna(1.0).max())
    fleet_cache_pct = 100.0 * (tot_cache / max(tot_compressed, 1e-6))
    wire_vs_logical_pct = 100.0 * (1.0 - (tot_wire / max(tot_logical_scan, 1e-6)))
    wire_vs_table_pct = 100.0 * (1.0 - (tot_wire / max(tot_table_logical, 1e-6)))

    providers = sorted({str(x) for x in df_jobs["remote_cloud"].dropna() if str(x)})
    providers_str = ", ".join(providers) if providers else "REMOTE"

    wire_card_bg = "#e6f4ea" if tot_wire < 0.05 or fleet_cache_pct >= 50.0 else "#fef7e0"
    wire_card_border = "#34a853" if tot_wire < 0.05 or fleet_cache_pct >= 50.0 else "#fbbc04"
    wire_card_fg = "#0d652d" if tot_wire < 0.05 or fleet_cache_pct >= 50.0 else "#8a4b00"

    body_rows = []
    for _, r in df_jobs.iterrows():
        c_pct = float(r["cache_hit_pct"] or 0.0)
        c_mb = float(r["cache_hit_mb"] or 0.0)
        w_mb = float(r["sent_over_wire_mb"] or 0.0)
        cache_style = "color:#0d652d;font-weight:700;" if c_pct > 0 else "color:#64748b;"
        wire_style = (
            "background:#e6f4ea;color:#0d652d;font-weight:700;padding:2px 8px;border-radius:999px;"
            if w_mb == 0.0
            else "color:#0f172a;font-weight:700;"
        )
        body_rows.append(
            f'<tr style="border-bottom:1px solid #e2e8f0;">'
            f'<td style="padding:8px 10px;font-weight:700;color:#334155;white-space:nowrap;">{html.escape(str(r["query_num"]))}</td>'
            f'<td style="padding:8px 10px;"><div style="font-weight:600;color:#0f172a;">{html.escape(str(r["query_title"] or ""))}</div>'
            f'<div style="font-size:11px;color:#64748b;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;margin-top:2px;">{html.escape(str(r["actual_physical_tables_read"] or ""))}</div></td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;">'
            f'<b>{float(r["elapsed_sec"] or 0):.2f}s</b><br><span style="font-size:11px;color:#475569;">{float(r["slot_sec"] or 0):,.1f}s (~{float(r["avg_slots"] or 0):,.1f} slots)</span></td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;color:#64748b;">{float(r["table_logical_mb"] or 0):,.2f} MB</td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;font-weight:600;color:#1967d2;">{float(r["logical_scan_mb"] or 0):,.2f} MB</td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;">'
            f'<span style="font-weight:600;color:#7627bb;">{float(r["compressed_mb"] or 0):,.2f} MB</span> '
            f'<span style="font-size:11px;background:#f3e8fd;color:#7627bb;padding:1px 6px;border-radius:6px;font-weight:700;">{html.escape(str(r["compression_ratio"]))}</span></td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;{cache_style}">{c_mb:,.2f} MB <span style="font-size:11px;">({c_pct:.1f}%)</span></td>'
            f'<td style="padding:8px 10px;text-align:right;white-space:nowrap;font-variant-numeric:tabular-nums;"><span style="{wire_style}">{w_mb:,.2f} MB</span> '
            f'<span style="font-size:10.5px;background:#f1f5f9;color:#334155;padding:2px 6px;border-radius:4px;font-weight:700;">{html.escape(str(r["remote_cloud"]))}</span></td>'
            f"</tr>"
        )

    display(HTML(f'''
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                background:#ffffff;color:#0f172a;border:1px solid #cbd5e1;border-radius:12px;
                padding:18px;margin:8px 0;box-shadow:0 1px 3px rgba(0,0,0,0.06);">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;margin-bottom:14px;">
        <div style="font-size:15px;font-weight:700;color:#0f172a;">
          📡 Cross-Cloud Data Reduction Waterfall Across {len(df_jobs)} Parallel Queries
          <span style="font-weight:500;font-size:13px;color:#475569;">({tot_slot_sec:,.1f} slot-sec executed in <code>{html.escape(LOCATION)}</code>)</span>
        </div>
        <div style="font-size:12px;background:#f1f5f9;color:#0f172a;padding:4px 10px;border-radius:999px;font-weight:600;">
          Remote Storage Provider: <code>{html.escape(providers_str)}</code>
        </div>
      </div>

      <div style="display:flex;align-items:stretch;gap:8px;flex-wrap:wrap;margin-bottom:16px;">
        <div style="flex:1;min-width:190px;background:#f8fafc;border:1px solid #93c5fd;border-top:4px solid #1967d2;border-radius:10px;padding:12px 14px;">
          <div style="font-size:11.5px;font-weight:700;text-transform:uppercase;letter-spacing:0.4px;color:#1967d2;">1. Logical Data Scanned</div>
          <div style="font-size:24px;font-weight:800;color:#0f172a;margin:4px 0;font-variant-numeric:tabular-nums;">{tot_logical_scan:,.2f} MB</div>
          <div style="font-size:11.5px;color:#475569;">
            Column-pruned <b>-{col_prune_pct:.1f}%</b> vs. <b>{tot_table_logical:,.1f} MB</b> full referenced tables
          </div>
        </div>

        <div style="display:flex;align-items:center;font-size:20px;font-weight:800;color:#94a3b8;">→</div>

        <div style="flex:1;min-width:190px;background:#faf5ff;border:1px solid #d8b4fe;border-top:4px solid #7627bb;border-radius:10px;padding:12px 14px;">
          <div style="font-size:11.5px;font-weight:700;text-transform:uppercase;letter-spacing:0.4px;color:#7627bb;">2. Compressed Data Read</div>
          <div style="font-size:24px;font-weight:800;color:#0f172a;margin:4px 0;font-variant-numeric:tabular-nums;">{tot_compressed:,.2f} MB</div>
          <div style="font-size:11.5px;color:#475569;">
            <b>{fleet_comp_ratio:.2f}x</b> avg compression ratio (up to <b>{peak_comp_ratio:.2f}x</b> on columnar scans)
          </div>
        </div>

        <div style="display:flex;align-items:center;font-size:20px;font-weight:800;color:#94a3b8;">→</div>

        <div style="flex:1;min-width:190px;background:#f0fdf4;border:1px solid #86efac;border-top:4px solid #137333;border-radius:10px;padding:12px 14px;">
          <div style="font-size:11.5px;font-weight:700;text-transform:uppercase;letter-spacing:0.4px;color:#137333;">3. Cross-Cloud Cache Hit</div>
          <div style="font-size:24px;font-weight:800;color:#0f172a;margin:4px 0;font-variant-numeric:tabular-nums;">{tot_cache:,.2f} MB</div>
          <div style="font-size:11.5px;color:#475569;">
            <b>{fleet_cache_pct:.1f}%</b> served from local <code>{html.escape(LOCATION)}</code> block cache
          </div>
        </div>

        <div style="display:flex;align-items:center;font-size:20px;font-weight:800;color:#94a3b8;">→</div>

        <div style="flex:1;min-width:205px;background:{wire_card_bg};border:1px solid {wire_card_border};border-top:4px solid {wire_card_border};border-radius:10px;padding:12px 14px;">
          <div style="font-size:11.5px;font-weight:700;text-transform:uppercase;letter-spacing:0.4px;color:{wire_card_fg};">4. Sent Over the Wire</div>
          <div style="font-size:24px;font-weight:800;color:{wire_card_fg};margin:4px 0;font-variant-numeric:tabular-nums;">{tot_wire:,.2f} MB</div>
          <div style="font-size:11.5px;color:{wire_card_fg};">
            Transferred from <b>{html.escape(providers_str)}</b> (<b>-{wire_vs_logical_pct:.1f}%</b> vs. logical / <b>-{wire_vs_table_pct:.1f}%</b> vs. full tables)
          </div>
        </div>
      </div>

      <details style="margin-bottom:12px;">
        <summary style="cursor:pointer;font-size:12px;color:#1a73e8;font-weight:600;">
          Show Executed Telemetry SQL (INFORMATION_SCHEMA.JOBS_BY_PROJECT + object_storage_stats)
        </summary>
        <pre style="background:#f8fafc;color:#0f172a;border:1px solid #e2e8f0;border-radius:6px;padding:10px;font-size:11.5px;overflow-x:auto;margin-top:6px;">{html.escape(REVEAL_JOBS_SQL.strip())}</pre>
      </details>

      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:12.5px;background:#ffffff;color:#0f172a;">
          <thead>
            <tr style="background:#f1f5f9;border-bottom:2px solid #cbd5e1;color:#334155;font-size:11.5px;">
              <th style="padding:8px 10px;text-align:left;">#</th>
              <th style="padding:8px 10px;text-align:left;">Query &amp; Physical Tables Resolved</th>
              <th style="padding:8px 10px;text-align:right;">Wall / Slots</th>
              <th style="padding:8px 10px;text-align:right;">Full Tables</th>
              <th style="padding:8px 10px;text-align:right;color:#1967d2;">1. Logical Scan</th>
              <th style="padding:8px 10px;text-align:right;color:#7627bb;">2. Compressed (Ratio)</th>
              <th style="padding:8px 10px;text-align:right;color:#137333;">3. Cache Hit</th>
              <th style="padding:8px 10px;text-align:right;">4. Sent Over Wire</th>
            </tr>
          </thead>
          <tbody>
            {"".join(body_rows)}
          </tbody>
          <tfoot>
            <tr style="background:#f8fafc;border-top:2px solid #94a3b8;font-weight:700;font-variant-numeric:tabular-nums;">
              <td colspan="2" style="padding:9px 10px;color:#0f172a;">BATCH TOTAL ({len(df_jobs)} Parallel Queries)</td>
              <td style="padding:9px 10px;text-align:right;color:#0f172a;">{tot_slot_sec:,.1f} slot-s</td>
              <td style="padding:9px 10px;text-align:right;color:#475569;">{tot_table_logical:,.2f} MB</td>
              <td style="padding:9px 10px;text-align:right;color:#1967d2;">{tot_logical_scan:,.2f} MB</td>
              <td style="padding:9px 10px;text-align:right;color:#7627bb;">{tot_compressed:,.2f} MB ({fleet_comp_ratio:.2f}x)</td>
              <td style="padding:9px 10px;text-align:right;color:#137333;">{tot_cache:,.2f} MB ({fleet_cache_pct:.1f}%)</td>
              <td style="padding:9px 10px;text-align:right;color:{wire_card_fg};">{tot_wire:,.2f} MB ({html.escape(providers_str)})</td>
            </tr>
          </tfoot>
        </table>
      </div>
    </div>
    '''))

---
## Knowledge Catalog
Inspect how **Knowledge Catalog** automatically discovers, indexes, and synthesizes **LLM-ready semantic context** (`:lookupContext`) across **federated external lakehouse tables (`@biglake`)** and **BigQuery datasets & ISO GQL Property Graphs (`@bigquery` / `ecommerce_graph`)** — with zero manual cataloging.

In [ ]:
# @title 8. Cataloged Assets & Agentic Context
_kc_sess = _kc_api_session()
_KC_BASE = "https://dataplex.googleapis.com/v1"

# 1. Search all auto-cataloged entries in this project across @biglake and @bigquery
_search_params = urllib.parse.urlencode({
    "query": f"(name:{CATALOG_ID} OR name:{GRAPH_DATASET_ID} OR system=BIGLAKE OR type=bigquery-graph)",
    "scope": f"projects/{PROJECT_ID}",
    "pageSize": 50,
})
_search_resp = _kc_sess.post(
    f"{_KC_BASE}/projects/{PROJECT_ID}/locations/global:searchEntries?{_search_params}",
    data=b"",
    timeout=30,
)
_search_err = ""
if not _search_resp.ok:
    try:
        _search_err = str(((_search_resp.json() or {}).get("error") or {}).get("message") or _search_resp.text)
    except Exception:
        _search_err = _search_resp.text
_search_json = _search_resp.json() if _search_resp.ok else {}
_search_results = _search_json.get("results") or []

_all_entry_names: list[str] = []
for _item in _search_results:
    _ename = ((_item.get("dataplexEntry") or {}).get("name")) or ""
    if _ename and f"/locations/{LOCATION}/" in _ename:
        if "/propertyGraphs/" in _ename and ("/nodes/" in _ename or "/edges/" in _ename):
            continue
        _all_entry_names.append(_ename)

def _entry_sort_key(ename: str) -> tuple[int, str]:
    if "/propertyGraphs/" in ename:
        return (0, ename)
    if "/@biglake/" in ename and "/tables/orders" in ename:
        return (1, ename)
    if "/@biglake/" in ename and "/tables/users" in ename:
        return (2, ename)
    if "/@biglake/" in ename and "/tables/events" in ename:
        return (3, ename)
    if "/@biglake/" in ename and "/tables/" in ename:
        return (4, ename)
    if "/@biglake/" in ename and "/catalogs/" in ename and "/namespaces/" not in ename:
        return (5, ename)
    if "/@biglake/" in ename and "/namespaces/" in ename:
        return (6, ename)
    if "/@bigquery/" in ename and "/datasets/" in ename:
        return (7, ename)
    return (8, ename)


_selected_entries = sorted(set(_all_entry_names), key=_entry_sort_key)[:11]


def _fetch_full_entry(entry_name: str) -> dict | None:
    q = urllib.parse.urlencode({"entry": entry_name, "view": "ALL"})
    r = _kc_sess.get(
        f"{_KC_BASE}/projects/{PROJECT_ID}/locations/{LOCATION}:lookupEntry?{q}",
        timeout=30,
    )
    return r.json() if r.ok else None


with ThreadPoolExecutor(max_workers=8) as _kc_pool:
    _detailed_entries = [e for e in _kc_pool.map(_fetch_full_entry, _selected_entries) if e]

# 2. Fetch LLM-ready YAML context via ':lookupContext' across a federated @biglake table + Property Graph
_ctx_targets: list[str] = []
for _e in _detailed_entries:
    _en = _e.get("name", "")
    if "/@biglake/" in _en and "/tables/" in _en and not any("/@biglake/" in x for x in _ctx_targets):
        _ctx_targets.append(_en)
    elif "/propertyGraphs/" in _en and not any("/propertyGraphs/" in x for x in _ctx_targets):
        _ctx_targets.append(_en)

_ctx_yaml = ""
if _ctx_targets:
    _ctx_r = _kc_sess.post(
        f"{_KC_BASE}/projects/{PROJECT_ID}/locations/{LOCATION}:lookupContext",
        json={"resources": _ctx_targets},
        timeout=30,
    )
    if _ctx_r.ok:
        _ctx_yaml = (_ctx_r.json() or {}).get("context", "")

_kc_search_url = f"https://console.cloud.google.com/dataplex/search?project={urllib.parse.quote(PROJECT_ID)}&q=thelook"
_kc_biglake_url = (
    f"https://console.cloud.google.com/dataplex/search?project={urllib.parse.quote(PROJECT_ID)}"
    f"&q={urllib.parse.quote('system=BIGLAKE')}"
)
_kc_graph_url = (
    f"https://console.cloud.google.com/dataplex/search?project={urllib.parse.quote(PROJECT_ID)}"
    f"&q={urllib.parse.quote('type=bigquery-graph ecommerce_graph')}"
)

if not _search_resp.ok:
    display(HTML(f'''
    <div style="padding:14px 16px;background:#fce8e6;color:#c5221f;border:1px solid #f28b82;border-radius:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
      <b>Knowledge Catalog API returned HTTP {_search_resp.status_code} on <code>{html.escape(PROJECT_ID)}</code></b>:
      <div style="margin-top:6px;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;word-break:break-word;">{html.escape(_search_err[:400])}</div>
    </div>
    '''))
elif not _detailed_entries:
    display(HTML(f'''
    <div style="padding:14px 16px;background:#fef7e0;color:#7c4a03;border:1px solid #fbbc04;border-radius:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
      <b>Knowledge Catalog entries are still indexing on <code>{html.escape(PROJECT_ID)}</code></b>.
      Wait ~30 seconds and re-run this cell, or open
      <a href="{html.escape(_kc_search_url)}" target="_blank">Knowledge Catalog Search ↗</a>.
    </div>
    '''))
else:
    _biglake_cnt = sum(1 for n in _all_entry_names if "/@biglake/" in n)
    _bq_cnt = sum(1 for n in _all_entry_names if "/@bigquery/" in n)
    _total_cols = 0
    _planes_seen: set[str] = set()
    _kc_rows_html: list[str] = []

    for _entry in _detailed_entries:
        _ename = _entry.get("name", "")
        _esrc = _entry.get("entrySource") or {}
        _aspects = _entry.get("aspects") or {}
        _short_name = _esrc.get("displayName") or _ename.rsplit("/", 1)[-1]
        _egroup = "@biglake" if "/@biglake/" in _ename else ("@bigquery" if "/@bigquery/" in _ename else "system")
        _etype_short = (_entry.get("entryType") or "").rsplit("/", 1)[-1]
        _sys_name = str(_esrc.get("system") or _egroup.lstrip("@").upper())
        _loc_name = str(_esrc.get("location") or LOCATION)

        _bl_data = _find_aspect_data(_aspects, "global.biglake-table")
        _st_data = _find_aspect_data(_aspects, "global.storage")
        _sc_data = _find_aspect_data(_aspects, "global.schema")

        _fields = _sc_data.get("fields") or []
        _total_cols += len(_fields)

        _uris = _st_data.get("sourceUris") or []
        _phys_uri = _uris[0] if _uris else (_esrc.get("resource") or _entry.get("fullyQualifiedName") or "")
        if _bl_data:
            _cat_src = str(_bl_data.get("catalogSource") or "EXTERNAL")
            _st_fmt = str(_bl_data.get("storageFormat") or _st_data.get("format") or "OPEN_TABLE")
            _engine_badge = f"{_cat_src} · {_st_fmt}"
            _badge_style = "background:#fff7ed;color:#c2410c;border:1px solid #fdba74;"
        elif "graph" in _etype_short:
            _engine_badge = "BIGQUERY · PROPERTY GRAPH"
            _badge_style = "background:#f0fdfa;color:#0f766e;border:1px solid #5eead4;"
        elif _egroup == "@biglake":
            _engine_badge = "BIGLAKE · FEDERATED CATALOG"
            _badge_style = "background:#fff7ed;color:#c2410c;border:1px solid #fdba74;"
        else:
            _engine_badge = f"BIGQUERY · {_etype_short.replace('bigquery-', '').upper()}"
            _badge_style = "background:#eff6ff;color:#1d4ed8;border:1px solid #93c5fd;"
        _planes_seen.add(_engine_badge)

        if _fields:
            _col_names = [str(f.get("name", "")) for f in _fields[:6] if isinstance(f, dict) and f.get("name")]
            _more = f" (+{len(_fields) - 6} more)" if len(_fields) > 6 else ""
            _meta_summary = f"{len(_fields)} columns: {', '.join(_col_names)}{_more}"
        elif "graph" in _etype_short:
            _meta_summary = "5 node labels, 4 edge labels, 6 semantic MEASURE(...) definitions"
        elif _esrc.get("description"):
            _meta_summary = str(_esrc.get("description"))
        else:
            _meta_summary = f"Auto-indexed {_etype_short} ({len(_aspects)} system aspect(s))"

        _entry_search_link = (
            f"https://console.cloud.google.com/dataplex/search?project={urllib.parse.quote(PROJECT_ID)}"
            f"&q={urllib.parse.quote(_short_name)}"
        )

        _kc_rows_html.append(f'''
        <tr style="border-bottom:1px solid #e2e8f0;vertical-align:top;">
          <td style="padding:9px 10px;white-space:nowrap;">
            <a href="{html.escape(_entry_search_link)}" target="_blank" rel="noopener noreferrer"
               style="font-weight:700;color:#1a73e8;text-decoration:none;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;">
              {html.escape(_short_name)} ↗
            </a>
            <div style="font-size:11px;color:#64748b;margin-top:2px;">EntryGroup: <code>{html.escape(_egroup)}</code></div>
          </td>
          <td style="padding:9px 10px;">
            <span style="display:inline-block;font-size:10.5px;font-weight:700;padding:2px 7px;border-radius:999px;{_badge_style}">
              {html.escape(_engine_badge)}
            </span>
            <div style="font-size:11px;color:#475569;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;margin-top:4px;word-break:break-all;">
              {html.escape(str(_phys_uri))}
            </div>
          </td>
          <td style="padding:9px 10px;white-space:nowrap;">
            <div style="font-weight:600;color:#0f172a;font-size:12px;"><code>{html.escape(_sys_name)}</code></div>
            <div style="font-size:11px;color:#64748b;margin-top:2px;">Region: <code>{html.escape(_loc_name)}</code></div>
          </td>
          <td style="padding:9px 10px;font-size:12px;color:#334155;line-height:1.45;min-width:260px;">
            {html.escape(_meta_summary)}
          </td>
        </tr>
        ''')

    _ctx_preview = _ctx_yaml.strip() or "# No ':lookupContext' YAML returned yet."
    _ctx_lines = len(_ctx_preview.splitlines())

    display(HTML(f'''
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                background:#ffffff;color:#0f172a;border:1px solid #cbd5e1;border-top:5px solid #4f46e5;
                border-radius:12px;padding:18px 20px;margin:8px 0;box-shadow:0 2px 6px rgba(79,70,229,0.08);">
      <div style="display:flex;justify-content:space-between;align-items:flex-start;flex-wrap:wrap;gap:12px;margin-bottom:14px;">
        <div>
          <div style="font-size:11.5px;font-weight:800;text-transform:uppercase;letter-spacing:0.6px;color:#4f46e5;">
            🧭 Knowledge Catalog — Automatic Discovery &amp; Agentic Context Plane
          </div>
          <div style="font-size:17px;font-weight:800;color:#0f172a;margin-top:3px;">
            Zero-Touch Discovery &amp; LLM Context Across Federated S3 Iceberg Tables &amp; ISO GQL Property Graphs
          </div>
        </div>
        <div style="display:flex;gap:8px;flex-wrap:wrap;">
          <a href="{html.escape(_kc_search_url)}" target="_blank" rel="noopener noreferrer"
             style="background:#4f46e5;color:#ffffff;text-decoration:none;font-weight:700;font-size:12px;padding:7px 12px;border-radius:8px;">
            🔎 Open Knowledge Catalog Search ↗
          </a>
          <a href="{html.escape(_kc_biglake_url)}" target="_blank" rel="noopener noreferrer"
             style="background:#fff7ed;color:#c2410c;border:1px solid #fdba74;text-decoration:none;font-weight:700;font-size:12px;padding:7px 12px;border-radius:8px;">
            Filter @biglake S3 Tables ↗
          </a>
          <a href="{html.escape(_kc_graph_url)}" target="_blank" rel="noopener noreferrer"
             style="background:#f0fdfa;color:#0f766e;border:1px solid #5eead4;text-decoration:none;font-weight:700;font-size:12px;padding:7px 12px;border-radius:8px;">
            Inspect ecommerce_graph Entry ↗
          </a>
        </div>
      </div>

      <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:10px;margin-bottom:16px;">
        <div style="background:#f8fafc;border:1px solid #cbd5e1;border-radius:10px;padding:11px 14px;">
          <div style="font-size:11px;font-weight:700;text-transform:uppercase;color:#475569;">Cataloged Assets Indexed</div>
          <div style="font-size:22px;font-weight:800;color:#0f172a;margin:3px 0;">{len(_all_entry_names)}</div>
          <div style="font-size:11.5px;color:#475569;"><b>{_biglake_cnt}</b> in <code>@biglake</code> · <b>{_bq_cnt}</b> in <code>@bigquery</code></div>
        </div>
        <div style="background:#eef2ff;border:1px solid #c7d2fe;border-radius:10px;padding:11px 14px;">
          <div style="font-size:11px;font-weight:700;text-transform:uppercase;color:#4338ca;">Storage-Agnostic Planes</div>
          <div style="font-size:22px;font-weight:800;color:#1e1b4b;margin:3px 0;">{len(_planes_seen)} Asset Types</div>
          <div style="font-size:11.5px;color:#3730a3;">Federated open tables + catalog + ISO GQL graph</div>
        </div>
        <div style="background:#f0fdf4;border:1px solid #86efac;border-radius:10px;padding:11px 14px;">
          <div style="font-size:11px;font-weight:700;text-transform:uppercase;color:#15803d;">Auto-Indexed Iceberg Columns</div>
          <div style="font-size:22px;font-weight:800;color:#14532d;margin:3px 0;">{_total_cols} Columns</div>
          <div style="font-size:11.5px;color:#166534;">Zero-touch <code>global.schema</code> &amp; <code>global.storage</code> aspects</div>
        </div>
        <div style="background:#f0fdfa;border:1px solid #5eead4;border-radius:10px;padding:11px 14px;">
          <div style="font-size:11px;font-weight:700;text-transform:uppercase;color:#0f766e;">Agentic Context (lookupContext)</div>
          <div style="font-size:22px;font-weight:800;color:#134e4a;margin:3px 0;">{_ctx_lines:,} YAML lines</div>
          <div style="font-size:11.5px;color:#115e59;">{len(_ctx_preview):,} chars grounded across storage &amp; graph</div>
        </div>
      </div>

      <details style="margin-bottom:14px;">
        <summary style="cursor:pointer;font-size:12.5px;color:#4f46e5;font-weight:700;">
          🤖 Inspect Live LLM-Ready Context Synthesized by Knowledge Catalog (<code>:lookupContext</code> YAML across <code>@biglake</code> + <code>ecommerce_graph</code>)
        </summary>
        <pre style="background:#0f172a;color:#e2e8f0;border-radius:8px;padding:12px 14px;font-size:11.5px;max-height:340px;overflow:auto;margin-top:8px;line-height:1.45;">{html.escape(_ctx_preview)}</pre>
      </details>

      <div style="overflow-x:auto;">
        <table style="width:100%;border-collapse:collapse;font-size:12.5px;background:#ffffff;color:#0f172a;">
          <thead>
            <tr style="background:#f1f5f9;border-bottom:2px solid #cbd5e1;color:#334155;font-size:11.5px;">
              <th style="padding:8px 10px;text-align:left;">Cataloged Entry</th>
              <th style="padding:8px 10px;text-align:left;">Engine / Format &amp; Physical Resource</th>
              <th style="padding:8px 10px;text-align:left;">System &amp; Region</th>
              <th style="padding:8px 10px;text-align:left;">Auto-Discovered Schema &amp; Topology</th>
            </tr>
          </thead>
          <tbody>
            {"".join(_kc_rows_html)}
          </tbody>
        </table>
      </div>
    </div>
    '''))